# Lead Scoring Model

This is a theoretical representation of the model; the parameters I will be using are:

1. Financial Qualification
2. Need, Problem and Product Fit
3. Authority and Decision Structure
4. Timeline, Urgency and Buying Stage
5. Engagement Behaviour
6. Company and Market Fit
7. Lead Source Quality
8. Competitive Landscape
9. Relationship and Trust Equity
10. Strategic and Lifetime Value

Within each parameter I introduce multiple variables to capture every potential scenario in the lead conversion journey. Each variable is assigned a specific weight to calculate a precise metric for that parameter. Finally, by weighting the parameters themselves, the model produces a robust, comprehensive score that accounts for most of the edge cases as well.

## Helper Functions

Before computing any parameter, we define a library of reusable mathematical tools that the scoring engine relies on throughout:

| Function | Purpose |
|---|---|
| `bci` (Bayesian Credible Interval) | Given observed successes and trials, returns a conservative lower-bound estimate from a Beta posterior, so that small sample sizes are automatically penalised. Used wherever a stated input is blended with historical evidence (budget confirmation, technical coverage, win rate, etc.). |
| `bci_from_rate` | Convenience wrapper that converts a rate plus sample size into a `bci` call. |
| `edp` (Exponential Decay Penalty) | Computes `exp(−k·x)` — a smooth, tuneable decay. Used for recency penalties (days since engagement), fiscal-year distance, compliance gaps, churn risk, and competitor-count suppression. |
| `ln_norm` (Logarithmic Normalisation) | Computes `log(1+count) / log(1+cap)` — maps an integer count into `[0, 1]` with diminishing returns. Used for stakeholder counts, engagement channel counts, use-case counts, and relationship tenure. |
| `sigmoid_transform` | A logistic curve centred at `x0` with steepness `k`. Compresses raw scores into a bounded `[0, 1]` range and amplifies separation around the decision boundary. Applied to budget ratio, engagement, company fit, and selected final scores. |
| `gaussian_penalty` | Computes `exp(−(x−ideal)² / 2σ²)` — penalises deviation from an ideal value with a bell-shaped curve, for variables where both "too high" and "too low" are undesirable. |
| `hrc` (Huber Robust Composite) | A robust weighted average: it computes an initial weighted mean, then iteratively down-weights any component whose value deviates from the composite by more than `δ`. This prevents a single outlier variable from disproportionately dragging a parameter score up or down. It is the primary aggregation function for every parameter. |
| `silverman_bandwidth` | Estimates the optimal kernel bandwidth from a historical score array using Silverman's rule. Used internally by `kpn`. |
| `kpn` (Kernel Percentile Normalisation) | Given a raw score and a distribution of historical scores, returns the percentile rank via Gaussian kernel density estimation, so a score can be interpreted relative to the population of past leads rather than in absolute terms. |
| `ewma_mean` (Exponentially Weighted Moving Average) | Weights recent observations more heavily than older ones (decay factor `λ`). Used to detect engagement momentum from a weekly activity series. |


In [168]:
import numpy as np
import math
from scipy.stats import beta as beta_dist, norm

def bci(successes, trials, confidence=0.05):
    if trials <= 0:
        return 0.0
    s = float(successes)
    n = float(trials)
    return beta_dist.ppf(confidence, s + 1, n - s + 1)
    
def bci_from_rate(rate, n):
    s = round(rate * n)
    return bci(s, n)
              
def edp(x, k):
    return np.exp(-k * x)
def ln_norm(count, cap):
    if cap <= 0:
        return 0.0
    return min(np.log(1 + count) / np.log(1 + cap), 1.0)
    
def sigmoid_transform(x, x0=0.50, k=8.0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))
    
def gaussian_penalty(actual, ideal, sigma):
    return np.exp(-((actual - ideal) ** 2) / (2 * sigma ** 2))
    
def hrc(values, weights, delta=0.15, max_iter=100, tol=1e-6):
    values = np.array(values, dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    theta = np.dot(weights, values)
    for _ in range(max_iter):
        residuals = np.abs(values - theta)
        adjusted_weights = weights.copy()
        for i in range(len(values)):
            if residuals[i] > delta:
                adjusted_weights[i] = weights[i] * (delta / residuals[i])
        if adjusted_weights.sum() == 0:
            break
        theta_new = np.dot(adjusted_weights, values) / adjusted_weights.sum()
        if abs(theta_new - theta) < tol:
            theta = theta_new
            break
        theta = theta_new
    return theta
    
def silverman_bandwidth(history):
    history = np.asarray(history, dtype=float)
    n = len(history)
    if n < 2:
        return 1.0
    sigma = np.std(history, ddof=1)
    q75, q25 = np.percentile(history, [75, 25])
    iqr = q75 - q25
    spread = min(sigma, iqr / 1.34) if iqr > 0 else sigma
    if spread <= 0:
        return 1.0
    return 0.9 * spread * (n ** (-1 / 5))
    
def kpn(raw_score, historical_scores):
    historical_scores = np.asarray(historical_scores, dtype=float)
    n = len(historical_scores)
    if n < 2:
        return raw_score 
    h = silverman_bandwidth(historical_scores)
    if h <= 0:
        return 0.5
    z = (raw_score - historical_scores) / h
    return float(np.mean(norm.cdf(z)))
    
def ewma_mean(values, lam=0.85):
    values = np.asarray(values, dtype=float)
    T = len(values)
    weights = np.array([lam ** (T - t - 1) for t in range(T)])
    weights /= weights.sum()
    return float(np.sum(weights * values))

In [1]:
data = [
    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 1: BUDGET & FINANCIAL READINESS
    # This section captures everything about the prospect's money situation:
    # how much they can spend, how confirmed that budget is, and how
    # complex their purchasing process will be.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "b": 50000,
        # ── Estimated Budget (Prospect's Available Money) ──
        # What it means:
        #   This is the total amount of money the prospect has set aside
        #   (or could realistically set aside) to buy our solution.
        # How to find it:
        #   - The prospect may directly tell you their budget.
        #   - If not, estimate it from clues like:
        #       • Their company size and revenue
        #       • Past purchases of similar products
        #       • What competitors in their industry typically spend
        #       • Their department's known spending patterns
        # How to fill it in:
        #   Enter a whole number in your curreancy (e.g., 50000 means $50,000).
        #   This is YOUR BEST ESTIMATE of what THEY can spend — not what
        #   we want to charge them.
        # Example:
        #   If a prospect says "we have around $50K for this project," enter 50000.
        #   If they haven't said anything but their company size suggests
        #   they could afford $30K–$60K, pick a reasonable midpoint like 45000.

        "d": 45000,
        # ── Deal Value (Our Required Price) ──
        # What it means:
        #   This is the total price WE need to charge for our solution.
        #   It includes everything the customer would pay us:
        #       • Software license or subscription fees
        #       • Implementation and setup costs
        #       • Onboarding and training fees
        #       • Ongoing service or support fees
        #       • Any add-ons or extras tied to this deal
        # How to fill it in:
        #   Enter a whole number in your currency (e.g., 45000 means $45,000).
        # Why it matters:
        #   If our deal value (d) is HIGHER than their budget (b), the deal
        #   is at risk — they may not be able to afford us. If it's LOWER,
        #   we're in a comfortable position.
        # Example:
        #   If our quote for everything is $45,000, enter 45000.

        "c": 0.75,
        # ── Budget Confirmation Level ──
        # What it means:
        #   How confident are we that the budget number (b) above is real
        #   and accurate? This is NOT about whether they have enough money —
        #   it's about how RELIABLY we know the number.
        # How to fill it in:
        #   Pick a value from 0 to 1 based on the strongest evidence you have:
        #
        #   0.00 = No confirmation at all
        #           We have zero information about their budget. We're
        #           purely guessing based on company size or industry norms.
        #
        #   0.25 = Light confirmation (a range was mentioned)
        #           The prospect casually said something like "we're probably
        #           looking at $40K to $60K" but nothing formal. It's a hint,
        #           not a commitment.
        #
        #   0.50 = Verbal confirmation (spoken but not written)
        #           The prospect told us on a call or in a meeting: "Our budget
        #           is $50,000." We trust what they said, but there's nothing
        #           in writing to back it up.
        #
        #   0.75 = Written confirmation (email or documented)
        #           The prospect confirmed the budget in an email, a chat
        #           message, or a shared document. We have a paper trail,
        #           but it's not an official procurement document.
        #
        #   1.00 = Formal procurement document
        #           The budget is confirmed in an official document — a
        #           Purchase Order (PO), a signed budget approval, an RFP
        #           with a stated budget, or a formal procurement form.
        #           This is the gold standard.
        #
        # Example:
        #   If the prospect emailed saying "We've earmarked $50K for this,"
        #   enter 0.75.

        "t": 3,
        # ── Fiscal Year Alignment (Months Until Budget Expires) ──
        # What it means:
        #   How many MONTHS away is the prospect's budget availability from
        #   our current sales timeline? A deal is easiest when the prospect's
        #   budget is available RIGHT NOW in their current fiscal cycle. It
        #   gets harder when the money depends on a future budget period
        #   that hasn't started yet.
        # How to fill it in:
        #   Enter the number of months until the prospect's relevant budget
        #   cycle begins or expires.
        #       • 0 = Budget is available right now, in their current fiscal period.
        #       • 3 = Budget becomes available in 3 months (e.g., next quarter).
        #       • 6 = Budget is 6 months away (e.g., next half-year cycle).
        #       • 12+ = Budget depends on next fiscal year or later.
        #   Internally, the system converts this to a 0–1 score using the
        #   formula: 1 - (T / 12). So 0 months = score of 1.0 (perfect
        #   alignment), 6 months = 0.5, 12 months = 0.0.
        # Example:
        #   If the prospect's fiscal year resets in 3 months and they need
        #   the new budget to buy, enter 3.

        "f2": 0.7,
        # ── Funding Source Type ──
        # What it means:
        #   How secure and formally approved is the SOURCE of the prospect's
        #   money? Even if a prospect says "we have budget," the money could
        #   come from a shaky source (like a manager's discretionary fund)
        #   or a rock-solid source (like a board-approved capital project).
        # How to fill it in:
        #   Pick a value from 0 to 1:
        #
        #   0.30 = No identified funding source
        #           The prospect hasn't told us where the money would come
        #           from. They might want our product, but there's no clear
        #           pot of money assigned to it yet.
        #
        #   0.50 = Departmental discretionary budget
        #           A department head or manager can spend this from their
        #           own flexible budget without needing higher approval.
        #           It's real money, but it can be redirected at any time
        #           if priorities change.
        #
        #   0.70 = Allocated project budget
        #           The money has been specifically earmarked for a defined
        #           project (e.g., "CRM upgrade project — $50K"). It's more
        #           committed than discretionary funds, but hasn't gone
        #           through the highest level of formal approval.
        #
        #   1.00 = Board-approved capital expenditure
        #           The funding was formally approved at the executive or
        #           board level as a capital investment. This is the most
        #           secure type of funding — it's extremely unlikely to be
        #           pulled or redirected.
        #
        # Example:
        #   If the prospect says "this is part of our digital transformation
        #   project budget," enter 0.7.

        "m": 0.5,
        # ── Multi-Year / Long-Term Contract Willingness ──
        # What it means:
        #   How open is the prospect to signing a contract that lasts longer
        #   than one year? Long-term contracts give us revenue stability;
        #   short-term or month-to-month deals carry the risk that the
        #   customer leaves after the initial period.
        # How to fill it in:
        #   Pick a value from 0 to 1:
        #
        #   0.0 = Refuses any long-term commitment
        #         The prospect only wants a one-time purchase, a short pilot
        #         project, or a month-to-month subscription. They will NOT
        #         consider locking in for multiple years. High risk that
        #         they leave after the initial period.
        #
        #   0.5 = Open to it, but it depends on negotiation
        #         The prospect hasn't ruled out a multi-year deal, but they
        #         want to see: better pricing/discounts for committing longer,
        #         service-level guarantees (SLAs), or proof of value during
        #         a trial period first. This is an invitation for our sales
        #         team to negotiate and make a compelling case.
        #
        #   1.0 = Actively seeking a long-term partnership
        #         The ideal scenario. The prospect WANTS a strategic partner
        #         for 2–3+ years. They value stability, locked-in pricing,
        #         and a deep vendor relationship. This deal represents high
        #         financial predictability for us.
        #
        # Example:
        #   If the prospect said "we'd consider a 2-year deal if the pricing
        #   makes sense," enter 0.5.

        "p": 0.75,
        # ── Procurement Complexity ──
        # What it means:
        #   How difficult is the prospect's internal buying process? Some
        #   companies can buy with a credit card in minutes; others require
        #   months of legal reviews, committee approvals, and formal bidding.
        #   Higher complexity = lower chance of the deal closing smoothly.
        # How to fill it in:
        #   Pick a value from 0.5 to 1 (NOTE: higher = easier):
        #
        #   1.00 = Extremely simple (credit card / click-to-buy)
        #          The prospect can purchase using a corporate credit card
        #          or a simple online agreement. No formal bidding, no
        #          complex legal reviews, zero administrative overhead.
        #          The deal can close in days.
        #
        #   0.75 = Moderate complexity (internal reviews required)
        #          The deal requires passing an internal IT security review,
        #          signing a custom Master Services Agreement (MSA), and
        #          setting up a formal Purchase Order (PO) through their
        #          finance department. Expect a 1-to-3-month cycle before
        #          the deal closes.
        #
        #   0.50 = Very complex (formal RFP / bidding process)
        #          The prospect forces us to go through a massive,
        #          competitive Request for Proposal (RFP) process. This
        #          involves strict compliance documentation, endless legal
        #          back-and-forth, background checks, and multiple committee
        #          sign-offs. The sales cycle could take 6 to 12+ months,
        #          with a high risk of the deal stalling entirely.
        #
        # Example:
        #   If the prospect said "we'll need to run this through our IT
        #   security team and set up a PO," enter 0.75.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 2: SOLUTION FIT & NEED
    # This section captures how well our product matches what the prospect
    # actually needs: their industry, their problem, their current tools,
    # and how much customisation would be required.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "r": 0.8,
        # ── Industry-Level Match ──
        # What it means:
        #   How closely does the prospect's industry match industries where
        #   our company ALREADY has proven success? If we've sold to 50
        #   healthcare companies and this prospect is in healthcare, that's
        #   a strong match. If this prospect is in a brand-new industry
        #   we've never sold to, it's a weak match.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely new/unrelated industry — we have zero
        #             experience or reference customers here.
        #       0.5 = Adjacent industry — somewhat related to our core
        #             industries, but limited direct experience.
        #       0.8 = Strong match — we have multiple successful customers
        #             in this exact industry.
        #       1.0 = Perfect match — this is one of our top-performing
        #             industries with extensive case studies and references.
        # Example:
        #   If we primarily serve fintech and the prospect is a fintech
        #   company, enter 0.9 or 1.0.

        "s": 0.9,
        # ── Solution Fit (Problem–Product Alignment) ──
        # What it means:
        #   How directly does our product solve the prospect's SPECIFIC
        #   problem or use case? This isn't about whether our product is
        #   good in general — it's about whether it solves THIS customer's
        #   particular pain point.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Our product has essentially nothing to do with their
        #             problem.
        #       0.3 = Loosely related — our product touches on their area
        #             but wasn't built for this use case.
        #       0.5 = Partial fit — solves some aspects of their problem
        #             but leaves major gaps.
        #       0.7 = Good fit — solves most of their problem with minor
        #             gaps or workarounds needed.
        #       0.9 = Excellent fit — our product was practically designed
        #             for this exact scenario.
        #       1.0 = Perfect fit — addresses every aspect of their stated
        #             problem.
        # Example:
        #   If the prospect needs automated invoice processing and our
        #   product's core feature is automated invoice processing, enter
        #   0.9 or 1.0.

        "p": 0.7,
        # ── Problem Statement Clarity ──
        # What it means:
        #   How clearly can the prospect explain the problem they're trying
        #   to solve? A prospect who can't articulate their problem is
        #   harder to sell to (they may not even know what they need). A
        #   prospect with a detailed, documented problem statement is much
        #   more likely to buy.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Extremely vague — they say things like "we want to
        #             improve things" or "we're exploring options" with no
        #             specifics at all.
        #       0.3 = Somewhat vague — they have a general sense of the
        #             problem area but can't describe specific pain points,
        #             impacts, or goals.
        #       0.5 = Moderate clarity — they can describe the problem
        #             verbally in reasonable detail but haven't documented
        #             it or quantified the impact.
        #       0.7 = Good clarity — they have a clear understanding of
        #             the problem with some supporting data or examples.
        #       1.0 = Fully documented — they have a detailed, written
        #             problem statement with measurable impacts, specific
        #             requirements, and clear success criteria.
        # Example:
        #   If the prospect says "We lose about 15 hours per week on manual
        #   data entry, and here's a spreadsheet showing the errors," enter
        #   0.7 or 0.8.

        "cs": 1,
        # ── Current Solution Exists ──
        # What it means:
        #   Does the prospect currently use another product, tool, or method
        #   to handle the problem they want us to solve? This tells us if
        #   we're replacing something or filling a gap that has nothing in
        #   place today.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No current solution at all — they are not using any
        #             tool, software, or process for this. It's a greenfield
        #             opportunity.
        #       0.5 = Partial/informal solution — they use spreadsheets,
        #             manual processes, free tools, or a workaround that
        #             wasn't designed for this purpose.
        #       1   = Yes, a formal solution exists — they currently use a
        #             dedicated product or vendor for this (e.g., a competitor's
        #             software, an in-house built tool, etc.).
        # Example:
        #   If the prospect currently uses a competitor's CRM, enter 1.
        #   If they track everything in spreadsheets, enter 0.5.

        "d": 0.6,
        # ── Dissatisfaction Level with Current Solution ──
        # What it means:
        #   How unhappy is the prospect with what they currently use? High
        #   dissatisfaction means they're more motivated to switch. Low
        #   dissatisfaction means they might not see enough reason to change.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Fully satisfied — they're happy with what they have.
        #             Very hard to get them to switch.
        #       0.3 = Mildly dissatisfied — a few annoyances, but nothing
        #             urgent. They'd consider alternatives if it were easy.
        #       0.5 = Moderately dissatisfied — noticeable pain points that
        #             affect productivity or results. Actively looking for
        #             something better.
        #       0.7 = Very dissatisfied — significant frustration, frequent
        #             complaints, and/or measurable business impact.
        #       1.0 = Deeply dissatisfied — the current solution is actively
        #             hurting their business. They urgently need to replace it.
        # Note:
        #   If "cs" above is 0 (no current solution), you can set this to
        #   a moderate value like 0.5 to represent their dissatisfaction
        #   with having NO solution (i.e., the pain of doing nothing).
        # Example:
        #   If the prospect says "Our current tool crashes weekly and we've
        #   lost data twice," enter 0.8 or 0.9.

        "t": 0.85,
        # ── Technical Fit (Out-of-the-Box Coverage) ──
        # What it means:
        #   What percentage of the prospect's stated requirements can our
        #   product handle IMMEDIATELY — right out of the box — without
        #   any custom development, special configuration, or workarounds?
        # How to fill it in:
        #   Enter a number from 0 to 1 (think of it as a percentage):
        #       0.0  = Our product covers 0% of their requirements as-is.
        #       0.5  = Our product covers about 50% of their needs out of
        #              the box; the rest would need custom work.
        #       0.85 = Our product covers 85% of their requirements with
        #              standard features.
        #       1.0  = Our product covers 100% of their requirements with
        #              no customisation needed at all.
        # Example:
        #   If the prospect listed 10 requirements and our product handles
        #   8 of them natively, enter 0.8.

        "c": 0.2,
        # ── Customisation Effort Required ──
        # What it means:
        #   How much additional custom development, special configuration,
        #   or bespoke work would be needed to fully meet the prospect's
        #   requirements? This is the flip side of technical fit — it
        #   captures the EFFORT and COST of closing the gap.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Zero customisation needed — everything works out of
        #             the box.
        #       0.2 = Minor customisation — small configuration tweaks,
        #             simple API integrations, or minor UI adjustments.
        #       0.5 = Moderate customisation — requires dedicated developer
        #             time, custom integrations, or workflow modifications.
        #       0.8 = Heavy customisation — significant engineering effort,
        #             new feature development, or major architectural changes.
        #       1.0 = Essentially a custom-built solution — the product
        #             would need to be fundamentally altered.
        # Example:
        #   If we just need to build one custom API connector, enter 0.2.

        "c2": 1.0,
        # ── Compliance Fit ──
        # What it means:
        #   Does our product meet the legal, regulatory, security, and
        #   industry compliance requirements that the prospect MUST follow
        #   before they're even ALLOWED to buy or use our solution?
        #   In many industries (healthcare, finance, government), a customer
        #   may love the product and have budget — but CANNOT purchase it
        #   unless it passes mandatory compliance checks (e.g., HIPAA, SOC 2,
        #   GDPR, FedRAMP, ISO 27001, etc.).
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0.0 = Non-compliant — our product does NOT meet their
        #             mandatory regulatory requirements. This is often a
        #             deal-breaker.
        #       0.5 = Partially compliant — we meet some requirements but
        #             not all. There may be a path to compliance, but it
        #             requires additional work or certifications.
        #       1.0 = Fully compliant — our product meets ALL of their
        #             required compliance, security, and regulatory
        #             standards. No blockers.
        # Example:
        #   If the prospect requires SOC 2 Type II and GDPR compliance, and
        #   we have both certifications, enter 1.0. If we have SOC 2 but
        #   not GDPR, enter 0.5.

        "u": 4,
        # ── Number of Use Cases Articulated ──
        # What it means:
        #   How many DISTINCT use cases (specific ways they'd use our product)
        #   has the prospect described to us? More use cases = more ways
        #   they see value in our product, which indicates deeper interest
        #   and a more mature buying decision.
        # How to fill it in:
        #   Count the number of separate, specific use cases the prospect
        #   has mentioned. Enter a whole number (integer).
        #       1 = They've mentioned only one way they'd use the product.
        #       3 = They've described three distinct use cases.
        #       5+ = They see our product solving many different problems
        #            across their organisation — very strong signal.
        # What counts as a "use case":
        #   Each distinct scenario or workflow where they'd apply our product.
        #   For example, if they say:
        #     (1) "We'd use it for customer onboarding"
        #     (2) "We'd also use it for internal training"
        #     (3) "And for compliance documentation"
        #   That's 3 use cases.
        # Example:
        #   If the prospect described 4 different ways they'd use the product,
        #   enter 4.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 3: AUTHORITY & DECISION-MAKING
    # This section captures who we're talking to, how much power they have,
    # and whether we have access to the people who actually make the final
    # buying decision.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "r": 0.8,
        # ── Primary Contact's Role Level (Seniority) ──
        # What it means:
        #   How senior is the MAIN person we're communicating with inside
        #   the prospect's organisation? Higher seniority usually means:
        #     • Stronger influence over the buying decision
        #     • Better visibility into company priorities and budget
        #     • Faster access to other decision-makers
        # How to fill it in:
        #   Pick ONE of these values:
        #       0.2 = Individual contributor / end user
        #             (e.g., analyst, developer, coordinator)
        #             They'll use the product but have little buying power.
        #       0.4 = Team lead / supervisor
        #             (e.g., team lead, senior specialist)
        #             Some influence, but not a budget holder.
        #       0.6 = Manager / department head
        #             (e.g., marketing manager, IT director)
        #             Controls a team budget and can champion the purchase.
        #       0.8 = Senior director / VP
        #             (e.g., VP of Sales, Director of Engineering)
        #             Significant authority, can approve mid-size purchases.
        #       1.0 = C-level / executive
        #             (e.g., CEO, CFO, CTO, COO)
        #             Ultimate decision-making authority.
        # Example:
        #   If our main contact is a VP of Operations, enter 0.8.

        "d": 1,
        # ── Decision Involvement of Our Contact ──
        # What it means:
        #   How directly does our primary contact influence or control the
        #   final purchase decision? Even a senior person might not be
        #   involved in THIS particular buying decision.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No involvement — they cannot influence the decision
        #             at all. They're just an information-gatherer or user
        #             who was assigned to talk to vendors.
        #       0.5 = Influencer — they provide input, recommendations, or
        #             evaluations, but someone else makes the final call.
        #       1   = Decision-maker — they have direct authority to approve
        #             or reject the purchase, or they are THE person who
        #             signs off on the deal.
        # Example:
        #   If our contact is the VP who will personally approve the
        #   purchase order, enter 1.

        "n": 4,
        # ── Total Number of Stakeholders ──
        # What it means:
        #   How many people inside the prospect's organisation are involved
        #   in (or need to approve) the purchase decision? This includes
        #   decision-makers, influencers, evaluators, legal reviewers,
        #   finance approvers — anyone who has a say.
        # How to fill it in:
        #   Enter a whole number (integer).
        #   More stakeholders generally means a more complex and slower
        #   sales process.
        #       1–2 = Simple decision — one or two people decide.
        #       3–5 = Moderate complexity — a small buying committee.
        #       6+  = Complex enterprise sale — many people involved,
        #             higher risk of delays and conflicting opinions.
        # Example:
        #   If the prospect has a VP, an IT manager, a procurement officer,
        #   and a legal reviewer involved, enter 4.

        "o": 0.8,
        # ── Organisational Alignment (Cross-Department Buy-In) ──
        # What it means:
        #   Are MULTIPLE departments or teams inside the prospect's
        #   organisation aligned on the need for our solution? When several
        #   departments agree they need the product, the deal is much
        #   stronger and harder to kill internally.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Only one person sees the need — no broader support.
        #       0.3 = One department supports it, but others are unaware
        #             or indifferent.
        #       0.5 = Two departments see value, but there's no formal
        #             cross-team agreement.
        #       0.8 = Multiple departments are aligned and actively
        #             supporting the purchase.
        #       1.0 = Organisation-wide alignment — this is a company-wide
        #             initiative with top-down support.
        # Example:
        #   If both the Sales team and the Marketing team are pushing for
        #   our product, enter 0.7 or 0.8.

        "p": 0.7,
        # ── Direct Access to Senior Stakeholders ──
        # What it means:
        #   How easily and directly can OUR sales team reach the senior
        #   stakeholders (executives, decision-makers) at the prospect's
        #   company? Sometimes we're stuck talking to a junior contact who
        #   "passes messages up." Other times we're directly in meetings
        #   with the C-suite.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No access at all — we're blocked from reaching
        #             senior people. Our contact won't introduce us upward.
        #       0.3 = Indirect access only — we communicate through our
        #             contact, who relays information to leadership.
        #       0.5 = Occasional access — we've had one or two brief
        #             interactions with a senior person, but it's not
        #             regular.
        #       0.7 = Good access — we can schedule meetings with senior
        #             stakeholders when needed. 
        #       1.0 = Full, direct access — we regularly communicate with
        #             the key decision-maker(s), and they're actively
        #             engaged in the evaluation.
        # Example:
        #   If we've had a couple of direct calls with the CTO and can
        #   email them directly, enter 0.7 or 0.8.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 4: TIMELINE & URGENCY
    # This section captures how quickly the prospect needs to make a
    # decision, what's driving their urgency, and what might slow things down.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "t": 45,
        # ── Days Until Decision ──
        # What it means:
        #   The estimated number of DAYS remaining before the prospect is
        #   expected to make a final purchase decision (yes or no).
        # How to fill it in:
        #   Enter a whole number (integer) representing calendar days.
        #       7  = They plan to decide within a week.
        #       30 = About a month out.
        #       45 = About six weeks.
        #       90 = About three months — a longer sales cycle.
        #       180+ = Very long cycle; the deal may stall.
        # Tip:
        #   Ask the prospect directly: "When are you looking to have a
        #   solution in place?" and work backward from their answer.
        # Example:
        #   If the prospect said "We want to make a decision by mid-July"
        #   and it's currently early June, enter 45.

        "t1": 1,
        # ── Trigger Event Happened ──
        # What it means:
        #   Has a specific event occurred that created urgency or accelerated
        #   the prospect's need to buy? Trigger events are things like:
        #     • A competitor launched a new product (competitive pressure)
        #     • Their current vendor announced end-of-life for their product
        #     • A new regulation takes effect on a specific date
        #     • They received new funding or budget approval
        #     • A key executive mandated the change
        #     • They experienced a major incident (security breach, outage)
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0 = No trigger event — they're evaluating at their own pace,
        #           with no external pressure forcing a decision.
        #       1 = Yes, a trigger event happened — something specific is
        #           driving urgency beyond normal interest.
        # Example:
        #   If their current tool's vendor announced they're shutting down
        #   in 6 months, enter 1.

        "t2": 30,
        # ── Days Until Trigger Event Deadline ──
        # What it means:
        #   If a trigger event happened (t1 = 1), how many DAYS until the
        #   deadline associated with that event? This creates a hard
        #   timeline the prospect can't easily push back.
        # How to fill it in:
        #   Enter a whole number (integer) representing calendar days.
        #   If no trigger event happened (t1 = 0), you can enter any value
        #   (it won't heavily impact scoring), but a reasonable default
        #   is to match the "days until decision" (t) or enter 0.
        # Example:
        #   If a new regulation takes effect in 30 days and the prospect
        #   must have a compliant solution by then, enter 30.

        "ep": 0.8,
        # ── Evaluation Process Clarity ──
        # What it means:
        #   How structured and clear is the prospect's process for evaluating
        #   and selecting a vendor? A well-defined process (with clear steps,
        #   criteria, and a timeline) means the deal is more likely to move
        #   forward predictably. A vague process means the deal could stall
        #   or disappear without warning.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No process at all — the prospect has no defined way
        #             of evaluating solutions. They're "just looking" with
        #             no structure.
        #       0.3 = Loosely defined — they have a general idea but no
        #             formal steps, criteria, or timeline.
        #       0.5 = Somewhat structured — they've outlined some steps
        #             (e.g., "we'll do demos, then decide") but lack
        #             formal evaluation criteria.
        #       0.8 = Well structured — they have a clear evaluation
        #             process with defined stages, a decision timeline,
        #             and evaluation criteria.
        #       1.0 = Formal, documented process — complete with an RFP,
        #             scoring rubric, defined decision committee, and a
        #             published timeline.
        # Example:
        #   If the prospect shared an evaluation timeline with demo dates
        #   and a decision date, enter 0.8.

        "ns": 1,
        # ── Next Step Defined ──
        # What it means:
        #   Is there a clearly agreed-upon NEXT ACTION scheduled between
        #   our team and the prospect? This is one of the strongest
        #   indicators of deal momentum. If there's always a clear next
        #   step, the deal is alive and moving. If there's no next step,
        #   the deal may be stalling.
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0 = No — there is no agreed-upon next step. We're waiting
        #           to hear back, or the conversation ended without
        #           scheduling anything.
        #       1 = Yes — there is a specific, scheduled next step (e.g.,
        #           a follow-up meeting on a set date, a demo scheduled,
        #           a proposal review call booked, a contract review
        #           session, etc.).
        # Example:
        #   If you have a demo scheduled for next Tuesday, enter 1.

        "cp": 0.3,
        # ── Competing Priorities ──
        # What it means:
        #   How much attention is the prospect dividing between THIS deal
        #   and OTHER business priorities? Even if they love our product,
        #   if they're juggling a merger, a product launch, and a
        #   restructuring, our deal might keep getting pushed down their
        #   to-do list.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = This is their TOP priority — they're focused almost
        #             entirely on solving this problem. Maximum attention
        #             and momentum.
        #       0.3 = Low distraction — this is a high priority for them,
        #             with only minor competing projects.
        #       0.5 = Moderate distraction — they have a few equally
        #             important projects, and our deal gets partial
        #             attention.
        #       0.7 = High distraction — many competing priorities, and
        #             our deal is not their top focus.
        #       1.0 = Extremely distracted — they are overwhelmed with
        #             other projects. Our deal is at serious risk of
        #             being deprioritised or forgotten.
        # Example:
        #   If the prospect mentioned they're also working on two other
        #   major initiatives but said ours is still a high priority,
        #   enter 0.3.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 5: ENGAGEMENT & ACTIVITY
    # This section tracks all the ways the prospect has interacted with us:
    # emails, calls, meetings, website visits, downloads, etc. More
    # engagement = stronger buying signals.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "n1": 10,
        # ── Email Opens ──
        # What it means:
        #   The total number of times the prospect has OPENED emails sent
        #   by our sales team. This is tracked by email tools (e.g.,
        #   HubSpot, Outreach, Salesloft). Each open counts — if they
        #   opened the same email 3 times, that counts as 3.
        # How to fill it in:
        #   Enter a whole number from your email tracking tool.
        #   If you don't track email opens, enter 0.
        # Example:
        #   If your CRM shows 10 total email opens from this prospect,
        #   enter 10.

        "n2": 5,
        # ── Email Replies ──
        # What it means:
        #   The total number of email RESPONSES the prospect has sent back
        #   to our team. Replies are a much stronger engagement signal
        #   than opens — they show the prospect is actively communicating
        #   with us.
        # How to fill it in:
        #   Enter a whole number. Count only replies FROM the prospect to
        #   our team, not our outbound emails.
        # Example:
        #   If the prospect has replied to 5 of our emails, enter 5.

        "n3": 3,
        # ── Meetings Completed ──
        # What it means:
        #   The number of scheduled meetings that were ACTUALLY COMPLETED
        #   with the prospect. This includes discovery calls, demo
        #   presentations, technical deep-dives, business reviews — any
        #   formal, scheduled meeting that both sides attended.
        # How to fill it in:
        #   Enter a whole number. Count only meetings that actually
        #   happened (not those that were scheduled but cancelled or
        #   no-showed).
        # Example:
        #   If you've had an intro call, a demo, and a technical review
        #   meeting, enter 3.

        "n4": 4,
        # ── Phone/Video Calls Completed ──
        # What it means:
        #   The number of meaningful phone calls or video calls completed
        #   with the prospect. These may overlap with meetings (n3) or
        #   be separate — for instance, a quick 10-minute phone check-in
        #   that wasn't a formal meeting. Count any substantive call where
        #   real business was discussed.
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If you've had 4 calls (including 2 formal demos and 2 informal
        #   check-ins), enter 4.

        "n5": 8,
        # ── Website Visits (Unique Sessions) ──
        # What it means:
        #   The number of unique website sessions by the prospect on OUR
        #   company website. This is tracked by analytics tools (e.g.,
        #   Google Analytics, HubSpot). If the prospect visited your site
        #   on Monday and again on Wednesday, that's 2 sessions.
        # How to fill it in:
        #   Enter a whole number from your website analytics or CRM.
        #   If you don't track this, enter 0.
        # Example:
        #   If your analytics show 8 website visits from this prospect's
        #   company, enter 8.

        "n6": 2,
        # ── Content Downloads ──
        # What it means:
        #   The number of downloadable resources the prospect has accessed
        #   from our website or sales materials. This includes:
        #     • Whitepapers or e-books
        #     • Case studies
        #     • Product datasheets
        #     • ROI calculators
        #     • Technical documentation
        #   Downloading content shows deeper interest than just visiting
        #   a webpage.
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect downloaded a case study and a product
        #   datasheet, enter 2.

        "n7": 3,
        # ── Pricing Page Visits ──
        # What it means:
        #   The number of times the prospect has visited our pricing page,
        #   packages page, or plan comparison page on our website. Pricing
        #   page visits are one of the strongest buying signals — they
        #   indicate the prospect is actively evaluating cost and thinking
        #   about purchasing.
        # How to fill it in:
        #   Enter a whole number from your website analytics.
        # Example:
        #   If the prospect visited the pricing page 3 separate times,
        #   enter 3.

        "n8": 1,
        # ── Demo/Trial Requests ──
        # What it means:
        #   The number of formal requests the prospect has made to see or
        #   test the product. This includes:
        #     • Requesting a live demo
        #     • Signing up for a free trial
        #     • Asking for a proof of concept (POC)
        #     • Requesting sandbox access
        #   This is a very high-intent action — they want hands-on
        #   experience with the product.
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect requested one product demo, enter 1.

        "n9": 2,
        # ── Social/Community Engagement ──
        # What it means:
        #   The number of interactions the prospect has had with our
        #   company through social media or community channels OUTSIDE
        #   of direct sales communication. This includes:
        #     • Liking, commenting, or sharing our LinkedIn posts
        #     • Engaging with us on Twitter/X
        #     • Participating in our community forum or Slack group
        #     • Attending our webinars or virtual events
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect commented on our LinkedIn post and attended
        #   a webinar, enter 2.

        "t": 7,
        # ── Days Since Last Meaningful Interaction ──
        # What it means:
        #   How many DAYS have passed since the last REAL, meaningful
        #   interaction with the prospect? "Meaningful" means a substantive
        #   exchange — a meeting, a detailed email reply, a phone call
        #   about the deal. An automated email open does NOT count.
        # How to fill it in:
        #   Enter a whole number (integer) representing calendar days.
        #       0–3  = Very recent — we just talked to them.
        #       7    = About a week ago.
        #       14   = Two weeks — starting to cool off.
        #       30+  = A month or more — the deal may be going cold.
        # Example:
        #   If the last real conversation was 7 days ago, enter 7.

        "v": 1.5,
        # ── Engagement Velocity ──
        # What it means:
        #   Is the prospect's engagement INCREASING or DECREASING over
        #   time? This is calculated as a ratio:
        #     (number of engagements in the LAST 14 days) ÷
        #     (number of engagements in the PRIOR 14 days)
        # How to interpret:
        #       < 1.0 = Engagement is SLOWING DOWN — they interacted less
        #               recently than before. Warning sign.
        #       = 1.0 = Engagement is STEADY — same level of activity.
        #       > 1.0 = Engagement is SPEEDING UP — they're becoming more
        #               active. Strong positive signal.
        # How to fill it in:
        #   Calculate or estimate the ratio. Enter a decimal number.
        #       0.5 = Engagement dropped by half (bad sign).
        #       1.0 = No change in engagement.
        #       1.5 = Engagement increased 50% (good sign).
        #       2.0 = Engagement doubled (excellent sign).
        # Example:
        #   If the prospect had 3 interactions in the prior 14 days and
        #   5 interactions in the last 14 days, enter 5/3 ≈ 1.67.

        "c": 4,
        # ── Channel Diversity ──
        # What it means:
        #   How many DIFFERENT communication channels is the prospect
        #   engaging with us through? Using multiple channels shows deeper
        #   and broader engagement.
        # How to count:
        #   Count each distinct channel as 1. Common channels:
        #     • Email
        #     • Phone
        #     • Video call (Zoom, Teams, etc.)
        #     • Website visits
        #     • Social media (LinkedIn, Twitter, etc.)
        #     • In-person meetings
        #     • Community forums / Slack
        #     • Events / webinars
        # How to fill it in:
        #   Enter a whole number.
        # Example:
        #   If the prospect communicates via email, has had phone calls,
        #   visits our website, and engages on LinkedIn, enter 4.

        "m": 1,
        # ── Negative Signals ──
        # What it means:
        #   The count of interactions that REDUCE our confidence or
        #   indicate resistance from the prospect. These are warning signs
        #   that the deal might be in trouble.
        # What counts as a negative signal:
        #     • Unsubscribing from our emails
        #     • Cancelling a scheduled meeting
        #     • No-showing to a meeting they confirmed
        #     • Saying "not interested" or "we're pausing this"
        #     • Significant delayed response after they committed to
        #       getting back to us by a certain date
        #     • Ghosting (going completely silent after active engagement)
        # How to fill it in:
        #   Enter a whole number. Count each negative event as 1.
        #       0 = No negative signals at all — great.
        #       1 = One minor negative event.
        #       3+ = Multiple warning signs — deal may be at risk.
        # Example:
        #   If the prospect cancelled one meeting, enter 1.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 6: IDEAL CUSTOMER PROFILE (ICP) FIT
    # This section checks how closely the prospect matches the TYPE of
    # customer your company is best at serving. Think of it as: "Does this
    # prospect LOOK like our best existing customers?"
    # ═══════════════════════════════════════════════════════════════════════
    {
        "seg": 0.9,
        # ── Segment Match ──
        # What it means:
        #   How closely does the prospect belong to the specific customer
        #   SEGMENT your company targets most successfully? A "segment"
        #   might be defined by industry vertical, company type, buyer
        #   persona, or business model (e.g., "mid-market B2B SaaS
        #   companies in North America").
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely outside our target segment.
        #       0.5 = Partially matches — overlaps with some aspects of
        #             our ideal segment but not all.
        #       0.9 = Very close match — fits our ideal customer profile
        #             almost perfectly.
        #       1.0 = Perfect match — this is exactly the type of customer
        #             we target and serve best.
        # Example:
        #   If our ideal customer is a mid-market SaaS company and this
        #   prospect is a mid-market SaaS company, enter 0.9 or 1.0.

        "emp": 0.8,
        # ── Employee Size Match ──
        # What it means:
        #   How closely does the prospect's number of employees match the
        #   company size that's ideal for our product? Some products work
        #   best for companies with 50–200 employees; others are built for
        #   enterprises with 10,000+.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Way too small or way too large for our product.
        #       0.5 = Somewhat outside our ideal size range, but could
        #             still work.
        #       0.8 = Close to our ideal employee size.
        #       1.0 = Perfectly within our ideal company size range.
        # Example:
        #   If our product works best for 200–1000 employee companies and
        #   the prospect has 500 employees, enter 0.9 or 1.0.

        "rev": 0.7,
        # ── Annual Revenue Match ──
        # What it means:
        #   How closely does the prospect's annual revenue match the
        #   revenue range of your ideal customer? This helps gauge whether
        #   they can afford our solution long-term and whether the deal
        #   size makes sense for our business.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Their revenue is far outside our ideal range (too
        #             small to afford us, or too large to care about us).
        #       0.5 = Somewhat outside our ideal range.
        #       0.7 = Close to our ideal revenue range.
        #       1.0 = Perfectly within our target revenue range.
        # Example:
        #   If our ideal customer has $10M–$50M revenue and this prospect
        #   has $30M, enter 0.9. If they have $5M, enter 0.6 or 0.7.

        "tech": 0.85,
        # ── Technology Stack Compatibility ──
        # What it means:
        #   How well does the prospect's EXISTING technology environment
        #   work with our product? Does our product integrate smoothly
        #   with the tools, systems, and platforms they already use?
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely incompatible — their tech stack doesn't
        #             work with our product at all.
        #       0.3 = Major compatibility issues — would require significant
        #             custom integration work.
        #       0.5 = Partially compatible — some integrations work, others
        #             don't.
        #       0.85 = Very compatible — most of their systems integrate
        #              smoothly with minor configuration.
        #       1.0 = Perfectly compatible — seamless integration with all
        #             their existing tools and systems.
        # Example:
        #   If the prospect uses Salesforce, AWS, and Slack, and our product
        #   integrates natively with all three, enter 0.9 or 1.0.

        "geo": 1.0,
        # ── Geographic Alignment ──
        # What it means:
        #   How well does the prospect's physical location fit within your
        #   company's active service regions? This matters for:
        #     • Legal/regulatory compliance (data residency laws)
        #     • Support coverage (time zones, language)
        #     • Sales team availability
        #     • On-site service requirements
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Completely outside our service area — we can't
        #             legally or practically serve them.
        #       0.3 = Fringe territory — we could serve them, but with
        #             significant logistical challenges.
        #       0.5 = Partially covered — some support limitations.
        #       0.8 = Well covered — within our active regions with minor
        #             constraints.
        #       1.0 = Perfect alignment — fully within our primary service
        #             region, time zone, and language coverage.
        # Example:
        #   If the prospect is based in a country where we have a local
        #   office and full support coverage, enter 1.0.

        "gro": 0.7,
        # ── Company Growth Trajectory ──
        # What it means:
        #   How fast is the prospect's company growing? A fast-growing
        #   company is more likely to expand their usage of our product
        #   over time (more seats, more features, higher tier).
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Declining — the company is shrinking (layoffs,
        #             revenue drops, market contraction).
        #       0.3 = Stagnant — flat growth, no significant changes.
        #       0.5 = Modest growth — growing slowly and steadily.
        #       0.7 = Good growth — expanding their team, revenue
        #             increasing, entering new markets.
        #       1.0 = Rapid growth — hypergrowth company, significant
        #             investment, fast hiring, rapidly expanding.
        # Example:
        #   If the prospect has been growing revenue 20% year-over-year
        #   and actively hiring, enter 0.7 or 0.8.

        "f": 0.8,
        # ── Financial Health / Credit Risk ──
        # What it means:
        #   How financially stable and low-risk is the prospect? A
        #   financially healthy company is more likely to pay on time,
        #   renew contracts, and expand. A financially struggling company
        #   might cancel, delay payments, or go bankrupt.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Severe financial distress — bankruptcy risk, unpaid
        #             debts, or known cash flow problems.
        #       0.3 = Concerning signs — recent layoffs, missed earnings,
        #             or negative press about financial health.
        #       0.5 = Average — nothing alarming, but no strong indicators
        #             of financial strength either.
        #       0.8 = Financially healthy — stable revenue, good credit,
        #             no red flags.
        #       1.0 = Excellent — well-funded, profitable, strong balance
        #             sheet, or backed by major investors.
        # Example:
        #   If the prospect is a profitable company with no known financial
        #   issues, enter 0.8.

        "d": 0.75,
        # ── Digital Readiness / Technology Adoption Readiness ──
        # What it means:
        #   How prepared is the prospect to adopt a digital tool like ours?
        #   Some companies are tech-savvy and quick to adopt new software.
        #   Others are still heavily manual, resistant to change, or lack
        #   the internal skills to implement new technology.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Very low readiness — the company is largely manual,
        #             resistant to technology, or lacks IT infrastructure.
        #       0.3 = Low readiness — they use basic tools but struggle
        #             with new technology adoption.
        #       0.5 = Moderate — they've adopted some digital tools but
        #             new implementations are slow and require heavy support.
        #       0.75 = Good readiness — tech-comfortable organisation with
        #              experience adopting similar tools.
        #       1.0 = Fully ready — digitally mature company that quickly
        #             adopts and integrates new technologies.
        # Example:
        #   If the prospect already uses modern cloud tools and has an IT
        #   team that manages implementations, enter 0.75 or 0.8.

        "l": 0.9,
        # ── Language & Culture Compatibility ──
        # What it means:
        #   How effectively can we WORK with this customer from a
        #   communication and business-culture perspective? This covers:
        #     • Shared language (or high English proficiency)
        #     • Compatible business norms and expectations
        #     • Similar communication styles
        #     • Time zone overlap for collaboration
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Major barriers — no shared language, completely
        #             different business culture, would require translators
        #             and cultural intermediaries.
        #       0.5 = Some challenges — we can communicate, but there are
        #             noticeable language or cultural friction points.
        #       0.9 = Very compatible — smooth communication, shared
        #             language, compatible work culture.
        #       1.0 = Perfect compatibility — same language, same business
        #             norms, same time zone, no friction at all.
        # Example:
        #   If the prospect speaks the same language and operates in a
        #   similar business culture to our team, enter 0.9 or 1.0.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 7: LEAD SOURCE QUALITY
    # This section captures HOW the prospect entered our pipeline and
    # the quality of the information we captured at that point. Better
    # sources = higher initial trust and intent.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "q": 0.75,
        # ── Source Channel Quality ──
        # What it means:
        #   How effective is the channel through which this lead first
        #   entered our sales pipeline? Some channels produce high-quality,
        #   high-intent leads; others produce low-quality leads that rarely
        #   convert. This is based on historical data about which channels
        #   work best for our company.
        # How to fill it in:
        #   Pick the value that matches how this lead found us (or how we
        #   found them). Listed from lowest to highest quality:
        #
        #   0.15 = Purchased list / cold outbound
        #          We bought their contact information or guessed their
        #          email. They've shown ZERO interest in our product. We're
        #          starting from zero trust and zero intent.
        #
        #   0.25 = Paid social ad (LinkedIn, Twitter/X, Facebook)
        #          They clicked an ad on social media. They fit our target
        #          audience, but they were originally on social media to
        #          scroll or network — not to solve a business problem.
        #
        #   0.35 = Paid search ad (Google Ads, Bing Ads)
        #          They typed a specific problem into a search engine and
        #          clicked our SPONSORED link. Shows active intent, but
        #          people naturally trust paid ads slightly less than
        #          organic results.
        #
        #   0.45 = Content marketing (blog, e-book, guide)
        #          They found and consumed our educational content. Trust
        #          is building, but they might be looking for free advice
        #          rather than a paid tool.
        #
        #   0.50 = Organic search
        #          They searched for a problem and clicked a NON-sponsored,
        #          organic link to our website. This shows both active
        #          intent and higher trust in our authority.
        #
        #   0.55 = Webinar attendee
        #          They committed an hour of their schedule to watch our
        #          presentation. Shows serious topic interest.
        #
        #   0.60 = Event / conference lead
        #          They interacted with us at a professional, industry-
        #          specific event — often face-to-face. Real-world
        #          interaction builds immediate trust.
        #
        #   0.70 = Free trial / freemium conversion
        #          They're already INSIDE our product. Intent is very high
        #          because they're actively testing whether our solution
        #          works for them. They just need to be convinced to pay.
        #
        #   0.75 = Partner referral
        #          A trusted business partner recommended us. The prospect
        #          transfers the trust they have in that partner directly
        #          to us.
        #
        #   0.85 = Inbound RFP (Request for Proposal)
        #          They sent us a formal RFP — meaning they have an active
        #          project, a timeline, and a budget. They are actively
        #          evaluating vendors and ready to write a check.
        #
        #   0.90 = Customer referral
        #          An existing, happy customer told this prospect to use
        #          our product. We don't need to prove credibility — the
        #          customer already did it for us. Highest-quality source.
        #
        # Example:
        #   If this lead came from a partner referral, enter 0.75.

        "p": 0.8,
        # ── Campaign / Asset Performance ──
        # What it means:
        #   How well did the specific marketing campaign or content asset
        #   (the ad, the webinar, the blog post, etc.) that generated this
        #   lead perform compared to our BEST-performing campaign ever?
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = The campaign performed terribly — very low conversion
        #             rates, poor lead quality.
        #       0.5 = Average campaign — performed normally compared to
        #             our other campaigns.
        #       0.8 = High-performing campaign — one of our better ones.
        #       1.0 = Our best-performing campaign — highest conversion
        #             rates and lead quality.
        # Tip:
        #   Your marketing team should be able to rank campaigns by
        #   conversion rate or lead-to-opportunity rate. Use that ranking
        #   to estimate this score.
        # Example:
        #   If this lead came from a webinar that had an 80% conversion
        #   rate (compared to our best at 100%), enter 0.8.

        "r": 0.7,
        # ── Data Richness at Point of Entry ──
        # What it means:
        #   How much information about the prospect did we capture at the
        #   moment they entered our pipeline? More data = better ability
        #   to qualify and personalise our outreach.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = We have almost nothing — just an email address.
        #       0.3 = Basic info — email + name.
        #       0.5 = Moderate info — email, name, company name, and
        #             maybe job title.
        #       0.7 = Good info — full profile including company, role,
        #             phone number, and some stated interest or context.
        #       1.0 = Complete profile — full contact details, company
        #             info, role, phone, stated problem/interest, budget
        #             range, and timeline. Everything we could want.
        # Example:
        #   If the prospect filled out a detailed form with their name,
        #   company, role, phone, and described their problem, enter 0.8
        #   or 0.9.

        "s": 1.0,
        # ── Inbound vs. Outbound ──
        # What it means:
        #   Did the prospect come to US (inbound), or did WE reach out to
        #   THEM (outbound)? Inbound leads generally have higher intent
        #   and trust because they initiated the relationship.
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0.4 = Outbound — WE reached out to the prospect first
        #             (cold email, cold call, outbound campaign, etc.).
        #             They didn't ask to hear from us.
        #       1.0 = Inbound — THEY came to us (filled out a form,
        #             requested a demo, replied to content, called us,
        #             etc.). They initiated the contact.
        # Example:
        #   If the prospect submitted a demo request on our website,
        #   enter 1.0. If we cold-emailed them, enter 0.4.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 8: COMPETITIVE LANDSCAPE
    # This section captures who else the prospect is considering, how
    # strong our position is against competitors, and how locked in the
    # prospect is with their current vendor.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "n": 3,
        # ── Number of Competitors Being Evaluated ──
        # What it means:
        #   How many OTHER vendors or alternative solutions is the prospect
        #   ACTIVELY considering alongside us? More competitors = more
        #   risk of losing the deal.
        # How to fill it in:
        #   Enter a whole number (integer).
        #       0 = We're the ONLY vendor they're evaluating (see "k" below).
        #       1 = They're looking at us + 1 other option.
        #       3 = They're comparing us against 3 other vendors.
        #       5+ = Highly competitive evaluation — many vendors in the
        #            running.
        # Note:
        #   This counts competitors ONLY — not us. If they're evaluating
        #   4 vendors total including us, enter 3.
        # Example:
        #   If the prospect mentioned they're also evaluating Competitor A,
        #   Competitor B, and Competitor C, enter 3.

        "w": 0.6,
        # ── Historical Win Rate Against These Competitors ──
        # What it means:
        #   Based on our past deals, what is our success rate when we
        #   compete against these SAME competitors in THIS customer segment?
        #   This comes from our CRM data — looking at past deals where we
        #   went head-to-head with the same competitors.
        # How to fill it in:
        #   Enter a number from 0 to 1 (think of it as a percentage):
        #       0.0 = We almost never win against these competitors.
        #       0.3 = We occasionally win — maybe 30% of the time.
        #       0.5 = It's a coin flip — we win about half the time.
        #       0.6 = We win more often than we lose.
        #       0.8 = We usually win against these competitors.
        #       1.0 = We almost always win when competing with them.
        # Tip:
        #   Ask your sales operations team: "What's our win rate against
        #   [Competitor X] in [this segment]?"
        # Example:
        #   If we win about 60% of deals when competing against these
        #   same vendors, enter 0.6.

        "s": 0.3,
        # ── Incumbent / Switching Stickiness ──
        # What it means:
        #   How strongly is the prospect already tied to their EXISTING
        #   vendor or current solution? A deeply entrenched incumbent
        #   makes it much harder for us to win the deal, even if our
        #   product is better.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No incumbent — they don't currently use any
        #             competing solution, or their current vendor has
        #             very weak hold on them.
        #       0.3 = Weak incumbent — they use something, but they're
        #             not deeply invested or contractually bound.
        #       0.5 = Moderate — they have a current vendor with some
        #             integration and history, but switching is feasible.
        #       0.7 = Strong incumbent — deep integration, long contract,
        #             significant relationship with the current vendor.
        #       1.0 = Deeply entrenched — long-term contract, massive
        #             integration, the current vendor is embedded in their
        #             daily workflows. Extremely hard to displace.
        # Example:
        #   If the prospect has a basic subscription to a competitor with
        #   no deep integration, enter 0.2 or 0.3.

        "d": 0.8,
        # ── Differentiation Strength ──
        # What it means:
        #   How clearly and convincingly can we explain WHY our solution
        #   is meaningfully different and BETTER than the alternatives
        #   for this prospect's specific needs? It's not about being
        #   better in general — it's about being better for THIS
        #   particular customer's situation.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = We have no meaningful differentiation — our product
        #             is essentially the same as competitors for this use
        #             case.
        #       0.3 = Weak differentiation — minor differences that aren't
        #             compelling enough to sway the decision.
        #       0.5 = Moderate — we have some unique features or advantages,
        #             but competitors have their own strengths too.
        #       0.8 = Strong differentiation — we have clear, compelling
        #             advantages that matter to this specific customer.
        #       1.0 = Dominant differentiation — we offer something no
        #             competitor can match for this customer's needs.
        # Example:
        #   If our product has a unique AI feature that directly addresses
        #   the prospect's biggest pain point, enter 0.8 or 0.9.

        "c": 0.4,
        # ── Switching Cost ──
        # What it means:
        #   How difficult and costly would it be for the prospect to switch
        #   FROM their current approach TO our solution? High switching
        #   costs are a barrier to adoption, even if our product is better.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Trivial to switch — they can start using our product
        #             tomorrow with minimal effort.
        #       0.2 = Low cost — minor data migration or configuration.
        #       0.4 = Moderate — some data transfer, team retraining, and
        #             process changes needed.
        #       0.7 = High cost — major data migration, extensive
        #             retraining, workflow redesign, and downtime.
        #       1.0 = Massive switching cost — complete system overhaul,
        #             months of migration, significant business disruption.
        # Example:
        #   If switching requires migrating data from their old system
        #   and retraining 50 users, enter 0.4 or 0.5.

        "k": 0,
        # ── Sole Vendor Status ──
        # What it means:
        #   Are we the ONLY vendor the prospect is currently considering?
        #   Being the sole vendor dramatically increases our chances of
        #   winning — there's no competition.
        # How to fill it in:
        #   Pick ONE of these two values:
        #       0 = No — other vendors are also being evaluated (see "n"
        #           above for how many).
        #       1 = Yes — we are the ONLY vendor being considered. The
        #           prospect is not looking at any alternatives.
        # Example:
        #   If the prospect told us "you're the only vendor we're talking
        #   to," enter 1. Otherwise, enter 0.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 9: RELATIONSHIP & TRUST
    # This section captures the history and depth of our relationship
    # with the prospect. Existing relationships and trust make deals
    # significantly easier to close.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "pb": 1.0,
        # ── Previous Business Relationship ──
        # What it means:
        #   Has our company had any prior business relationship with this
        #   prospect? Selling to an existing or past customer is very
        #   different from selling to a brand-new prospect — there's
        #   already a foundation of trust (or distrust).
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = Net new — we have NEVER done business with this
        #             prospect. They are a completely new lead.
        #       0.5 = Lapsed customer or informal history — we had a
        #             relationship in the past but it ended (they churned
        #             or the project concluded), OR we've had informal
        #             interactions (events, partnerships) but no formal
        #             business.
        #       1.0 = Active or recent customer — they currently buy from
        #             us, or they were a customer recently and the
        #             relationship is still warm.
        # Example:
        #   If this prospect is a current customer looking to buy an
        #   additional product, enter 1.0.

        "rt": 18,
        # ── Relationship Tenure (Months) ──
        # What it means:
        #   How long (in months) have we had ANY kind of relationship with
        #   this prospect? This includes the time they've been a customer,
        #   a partner, or even just a known contact in our CRM.
        # How to fill it in:
        #   Enter a whole number (integer) representing months.
        #       0  = Brand new — we just connected with them.
        #       6  = We've known them about 6 months.
        #       18 = About 1.5 years of relationship history.
        #       36 = 3 years of history.
        # Note:
        #   If "pb" above is 0 (net new), enter 0 here as well.
        # Example:
        #   If this prospect has been a customer for 18 months, enter 18.

        "nps": 0.5,
        # ── Prospect Sentiment (Net Promoter-Style Score) ──
        # What it means:
        #   What is the prospect's overall feeling or sentiment toward
        #   our company? This is inspired by the Net Promoter Score (NPS)
        #   concept — are they a fan (promoter), neutral (passive), or
        #   unhappy with us (detractor)?
        # How to fill it in:
        #   Enter a number from -1 to 1:
        #       -1.0 = Detractor — they actively dislike our company.
        #              They've had bad experiences and would warn others
        #              away from us.
        #       -0.5 = Mildly negative — some dissatisfaction or
        #              unresolved issues, but not hostile.
        #        0.0 = Neutral / passive — no strong feelings either way.
        #              This is the default for brand-new leads with no
        #              prior history.
        #        0.5 = Mildly positive — they have a favorable impression
        #              of us but aren't enthusiastic advocates.
        #        1.0 = Promoter — they love us and would actively
        #              recommend us to others.
        # Note:
        #   For completely new leads with no prior interaction, default
        #   to 0.
        # Example:
        #   If the prospect is a current customer who has given us
        #   positive feedback but hasn't gone out of their way to promote
        #   us, enter 0.5.

        "es": 0.5,
        # ── Executive Sponsor Relationship ──
        # What it means:
        #   How strong is our personal relationship with a SENIOR EXECUTIVE
        #   or decision-maker inside the prospect's organisation? Having
        #   a strong executive sponsor can dramatically accelerate a deal
        #   and help overcome internal obstacles.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No relationship — we have not connected with any
        #             senior executive at the prospect's company.
        #       0.5 = Acquaintance — we've met or spoken with an executive
        #             (e.g., at a conference, on a brief intro call), but
        #             the relationship is not deep or personal.
        #       1.0 = Strong personal relationship — we have an ongoing,
        #             trusted relationship with a key decision-maker.
        #             They take our calls, advocate for us internally,
        #             and actively support the deal.
        # Example:
        #   If we've had one intro call with the VP but don't have a
        #   deep relationship yet, enter 0.5.

        "t": 0.7,
        # ── Trust Indicator Level ──
        # What it means:
        #   How much observable evidence is there that the prospect TRUSTS
        #   us enough to be open, transparent, and collaborative? Trust
        #   is shown through actions, not words. Look for:
        #     • Sharing confidential or internal information with us
        #     • Introducing us to other stakeholders
        #     • Giving us early access to requirements or RFP drafts
        #     • Being transparent about budget, timeline, and competitors
        #     • Asking for our strategic advice (not just product info)
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No trust signals — the prospect is guarded, shares
        #             minimal information, and keeps us at arm's length.
        #       0.3 = Low trust — basic professional politeness but no
        #             real openness.
        #       0.5 = Moderate trust — they share some information and
        #             engage in honest conversation, but keep certain
        #             details private.
        #       0.7 = Good trust — they've shared internal documents,
        #             introduced us to colleagues, or been transparent
        #             about their decision process.
        #       1.0 = High trust — they treat us as a trusted advisor,
        #             share confidential plans, and proactively include
        #             us in strategic discussions.
        # Example:
        #   If the prospect shared their internal evaluation criteria and
        #   introduced us to their IT director, enter 0.7.

        "rc": 0.8,
        # ── Reference Customer Availability ──
        # What it means:
        #   Can we point to a successful, existing customer that this
        #   prospect would find RELEVANT and CREDIBLE? The best references
        #   are customers the prospect personally knows, or companies in
        #   the same industry/size that the prospect would respect.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No relevant references — we have no customers in
        #             their industry, size, or geography. We can't show
        #             them a relatable success story.
        #       0.3 = Weak reference — we have a customer in a loosely
        #             related industry, but it's not a strong match.
        #       0.5 = Moderate — we have a relevant customer but they're
        #             not in the exact same situation, or the prospect
        #             doesn't know them.
        #       0.8 = Strong reference — we have a successful customer
        #             in the same industry, similar size, or the prospect
        #             might know them by reputation.
        #       1.0 = Perfect reference — we have a successful customer
        #             that the prospect personally knows or deeply
        #             respects (e.g., a peer company, a partner, or a
        #             well-known brand in their space).
        # Example:
        #   If we have a case study from a well-known company in the
        #   prospect's industry, enter 0.8.

        "n": 0,
        # ── Previous Negative Experience ──
        # What it means:
        #   Has the prospect had a BAD past experience with our company?
        #   If yes, how serious was it? Past negative experiences create
        #   trust barriers that we need to address before the deal can
        #   move forward.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No negative experience — either they're a new
        #             prospect with no history, or their past experience
        #             was positive/neutral.
        #       0.5 = Minor negative experience — they had a frustrating
        #             incident (e.g., a support issue, a billing error, a
        #             minor product bug) that was eventually resolved but
        #             left a slightly sour taste.
        #       1   = Major negative experience — they had a serious
        #             problem with our company (e.g., a failed
        #             implementation, a major outage, broken promises,
        #             or a bad product experience). This creates
        #             significant resistance to buying from us again.
        # Example:
        #   If the prospect is a current customer with no complaints,
        #   enter 0.
    },

    # ═══════════════════════════════════════════════════════════════════════
    # CATEGORY 10: STRATEGIC VALUE
    # This section looks at the LONG-TERM value of winning this deal —
    # not just the initial sale, but the lifetime revenue, expansion
    # potential, and strategic benefits for our company.
    # ═══════════════════════════════════════════════════════════════════════
    {
        "ltv": 1500000,
        # ── Estimated Lifetime Value (LTV) ──
        # What it means:
        #   The total expected revenue our company will earn from this
        #   customer over the FULL DURATION of the relationship — not
        #   just the first deal, but all renewals, expansions, and
        #   upsells over the years.
        # How to calculate:
        #   LTV = (Annual contract value) × (Expected number of years
        #         they'll remain a customer) + (Expected expansion revenue)
        # How to fill it in:
        #   Enter a whole number in your currency.
        # Example:
        #   If the initial deal is $45K/year and you expect the customer
        #   to stay for 5 years with some expansion, resulting in a total
        #   of $300K over their lifetime, enter 300000. If the total
        #   expected lifetime revenue including expansions is $1.5M,
        #   enter 1500000.

        "cac": 400000,
        # ── Customer Acquisition Cost (CAC) ──
        # What it means:
        #   The total cost we're spending to acquire this customer. This
        #   includes all sales and marketing costs divided by the number
        #   of new customers acquired.
        #   Formula: (Total sales & marketing spend) ÷ (Number of new
        #            customers acquired in that period)
        # How to fill it in:
        #   Enter a whole number in your currency. If you don't know the
        #   exact number, use your company's average CAC.
        # Why it matters:
        #   Compare this to LTV. A healthy business has LTV much higher
        #   than CAC (typically 3× or more). If CAC is close to or higher
        #   than LTV, the deal may not be profitable.
        # Example:
        #   If our average cost to acquire a customer (sales team time,
        #   marketing spend, tools, etc.) is $400,000, enter 400000.

        "cs": 0.7,
        # ── Cross-Sell / Upsell Opportunity ──
        # What it means:
        #   How much opportunity exists to sell ADDITIONAL products or
        #   services to this customer AFTER the initial purchase? A
        #   customer who might buy more from us over time is worth more
        #   than one who will only ever buy one thing.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No cross-sell/upsell potential — this is likely a
        #             one-time purchase with no room for expansion.
        #       0.3 = Low potential — maybe one or two minor add-ons.
        #       0.5 = Moderate potential — a few additional products or
        #             tiers they could buy.
        #       0.7 = Good potential — multiple additional products,
        #             services, or higher tiers they'd likely need.
        #       1.0 = Massive potential — this customer could eventually
        #             buy our entire product suite and become one of our
        #             largest accounts.
        # Example:
        #   If we currently sell them a CRM and they could also buy our
        #   marketing automation and analytics tools, enter 0.7.

        "b": 0.6,
        # ── Brand Value of the Client ──
        # What it means:
        #   How valuable would it be to have this company as a customer
        #   from a REPUTATION and MARKETING perspective? Some customers
        #   are worth winning even at a discount because having their
        #   logo on your website opens doors to similar companies.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = No brand value — unknown company, no marketing
        #             benefit from winning them.
        #       0.3 = Minor brand value — a decent company, but not one
        #             that would impress other prospects.
        #       0.5 = Moderate — a recognisable company in their industry,
        #             useful as a reference.
        #       0.8 = High brand value — a well-known company whose logo
        #             would significantly boost our credibility.
        #       1.0 = Flagship account — a world-famous brand that every
        #             prospect in our market would recognise and respect.
        # Example:
        #   If winning this customer would give us a credible reference
        #   in a new industry, enter 0.6 or 0.7.

        "m": 0.5,
        # ── Market Entry / Market Expansion Value ──
        # What it means:
        #   Would winning this customer help us enter or strengthen our
        #   presence in a NEW market (new industry, new geography, new
        #   company size segment)? Strategic deals that open new markets
        #   have extra long-term value beyond the immediate revenue.
        # How to fill it in:
        #   Pick ONE of these three values:
        #       0   = No market entry value — this customer is in a market
        #             we already serve well. Winning them doesn't open
        #             any new doors.
        #       0.5 = Some market entry value — this deal could help us
        #             establish a foothold in a new segment, geography,
        #             or industry. It's not our primary market, but
        #             winning here could lead to more similar deals.
        #       1.0 = High market entry value — this is a strategic
        #             beachhead deal. Winning this customer would be our
        #             first success in an entirely new market that we
        #             want to expand into.
        # Example:
        #   If we've never sold to the healthcare industry and this
        #   prospect is a hospital, enter 0.5 or 1.0.

        "e": 0.65,
        # ── Expansion Revenue Probability ──
        # What it means:
        #   How likely is it that this customer will EXPAND their usage
        #   (and spending) after the initial purchase? Expansion can mean:
        #     • Adding more user seats
        #     • Upgrading to a higher tier
        #     • Buying additional modules or features
        #     • Rolling out to additional departments or locations
        # How to fill it in:
        #   Enter a number from 0 to 1 (think of it as a probability):
        #       0.0 = Very unlikely to expand — they'll use exactly what
        #             they buy, nothing more.
        #       0.3 = Low probability — some chance of minor expansion.
        #       0.5 = Moderate — about a 50/50 chance they'll expand.
        #       0.65 = Good probability — more likely than not that they'll
        #              grow their usage.
        #       1.0 = Almost certain to expand — strong signals that
        #             they'll scale up significantly after initial success.
        # Example:
        #   If the prospect said "we'd start with one department and
        #   potentially roll out company-wide," enter 0.65 or 0.7.

        "c": 0.2,
        # ── Churn Risk Indicator ──
        # What it means:
        #   How likely is it that this customer will STOP buying from us,
        #   cancel their subscription, or fail to renew in the future?
        #   High churn risk means the revenue may be short-lived, even
        #   if we win the deal.
        # How to fill it in:
        #   Enter a number from 0 to 1:
        #       0.0 = Very low churn risk — this customer is likely to
        #             stay for years. Strong fit, strong need, high
        #             switching cost.
        #       0.2 = Low risk — a few minor concerns, but overall a
        #             sticky customer.
        #       0.5 = Moderate risk — some factors that could lead to
        #             cancellation (e.g., budget uncertainty, weak fit,
        #             or available alternatives).
        #       0.7 = High risk — significant concerns about long-term
        #             retention.
        #       1.0 = Very high risk — strong likelihood they'll cancel
        #             within the first year (e.g., they only need a
        #             one-time solution, they're financially unstable, or
        #             our product is a poor long-term fit).
        # Example:
        #   If the customer has a strong need and high switching costs
        #   (making it hard for them to leave), enter 0.1 or 0.2.
    }
]


gates = ["f", "f", "f", "f", "f", "f", "f"]
modifiers = [1, 1, 0.9, 1, 1]

weights = [0.15, 0.12, 0.20, 0.12, 0.12, 0.08, 0.04, 0.05, 0.05, 0.07]


## Parameter 1 — Financial Qualification

This parameter assesses whether the prospect can realistically pay for the deal. It combines budget capacity, confirmation reliability, fiscal timing, funding security, long-term willingness, and procurement friction.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `B` | Estimated budget — the total amount the prospect has allocated or can realistically allocate for our solution. | currency |
| `D` | Deal value — our total quoted price (licence, implementation, subscription, fees). | currency |
| `Br` | Budget ratio = `min(B/D, 1)`. Ideal value → 1. | `[0, 1]` |
| `C` | Budget confirmation level — reliability of the budget figure. 0 = unconfirmed, 0.25 = range mentioned, 0.5 = verbal, 0.75 = written/email, 1 = formal procurement document. | `[0, 1]` |
| `F1` | Fiscal year alignment — how far out the prospect's buying cycle sits, computed via exponential decay `exp(−0.25·T_months)` so alignment degrades smoothly with distance. | `[0, 1]` |
| `F2` | Funding source type — security of the funding source. 0.3 = unidentified, 0.5 = departmental discretionary, 0.7 = allocated project budget, 1.0 = board-approved CapEx. | `[0, 1]` |
| `M` | Multi-year willingness — openness to contracts longer than one year. 0 = refuses, 0.5 = conditional, 1 = actively seeking multi-year. | `[0, 1]` |
| `P` | Procurement complexity — difficulty of the approval process. 1.0 = credit-card simple, 0.75 = MSA + PO, 0.5 = full RFP with committee sign-offs. | `[0.5, 1]` |

### How it is computed

- **Budget ratio** is passed through a sigmoid transform (`x0 = 0.70`, `k = 10`). This compresses ratios near 1.0 (diminishing returns once budget clearly covers the deal) and penalises ratios below ~0.70 more aggressively.
- **Budget confirmation** blends the stated level `C_stated` 50/50 with a Bayesian credible interval `C_evidence`, derived from `n_confirmed_budget` out of `n_similar_deals`. This anchors confidence to empirical conversion history rather than self-reported confirmation alone.
- **Fiscal alignment** uses exponential decay `exp(−0.25·T)`, giving a smooth degradation that never goes negative.
- All six components are aggregated with the Huber Robust Composite (`hrc`), so a single outlier variable cannot disproportionately inflate or deflate the parameter score.

### Composite Formula

```
X1 = HRC(
    values  = [Br_sigmoid, C_blended, F1_decay, F2, M, P],
    weights = [0.45, 0.17, 0.12, 0.12, 0.08, 0.06]
)
```


In [154]:
p1 = data[0]

br_raw = min(p1["b"] / p1["d"], 1.0)
Br = sigmoid_transform(br_raw, x0=0.70, k=10.0)

C_stated = p1["c"]
C_evidence = bci(p1["n_confirmed_budget"], p1["n_similar_deals"])
C = 0.5 * C_stated + 0.5 * C_evidence

F1 = edp(p1["t"], 0.25)

F2 = p1["f2"]

M_val = p1["m"]

P_val = p1["p"]

X1 = hrc(
    values=[Br, C, F1, F2, M_val, P_val],
    weights=[0.45, 0.17, 0.12, 0.12, 0.08, 0.06]
)

print(f"X1 (Financial Qualification): {X1:.6f}")


X1 (Financial Qualification): 0.783330


## Parameter 2 — Need, Problem & Product Fit

Evaluates how well our product matches the prospect's needs, technical environment, and compliance requirements.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `R` | Industry match — alignment with industries where we have proven success. | `[0, 1]` |
| `S` | Solution fit — how directly our product solves their stated problem. | `[0, 1]` |
| `P` | Problem clarity — how well the prospect can articulate the problem. 0 = vague, 1 = detailed and documented. | `[0, 1]` |
| `Cs` | Current solution exists — whether they already use a competing method. 0 = none, 0.5 = partial, 1 = full alternative in place. | `{0, 0.5, 1}` |
| `D` | Dissatisfaction with current solution. 0 = fully satisfied, 1 = deeply dissatisfied. | `[0, 1]` |
| `T` | Technical requirements coverage — share of stated requirements our product handles out-of-the-box. | `[0, 1]` |
| `C` | Custom work needed — extent of additional development or configuration required. | `[0, 1]` |
| `C2` | Compliance fit — whether our product meets the prospect's regulatory requirements. 0 = non-compliant, 0.5 = partial, 1 = fully compliant. | `{0, 0.5, 1}` |
| `U` | Number of distinct use cases the prospect has articulated (proxy for buyer intent and deal maturity). | integer |

### How it is computed

- **Technical coverage** (`T`) is computed via BCI from `n_features_available` out of `n_features_needed`, penalising high-coverage claims when the feature sample is small.
- **Positive fit** sub-score `F_pos` aggregates `[R, S, P, T_bci]` using `hrc` (weights 0.25, 0.30, 0.20, 0.25).
- **Negative fit** sub-score `F_neg` combines three drag terms via `hrc` (weights 0.40, 0.25, 0.35):
  - Dissatisfaction drag: `1 − 0.4·D·Cs`
  - Custom drag: `1 − 0.25·C`
  - Compliance gap via exponential decay `exp(−3.0·(1 − C2))` — a small gap barely hurts, a full gap is devastating.
- **Use-case count** is normalised via `ln_norm(U, 6)` (diminishing returns past 6).
- The three sub-scores are combined with `hrc` (weights 0.45, 0.40, 0.15).

### Composite Formula

```
F_pos   = HRC([R, S, P, T_bci], [0.25, 0.30, 0.20, 0.25])
F_neg   = HRC([1 − 0.4·D·Cs, 1 − 0.25·C, exp(−3·(1−C2))], [0.40, 0.25, 0.35])
U_score = ln_norm(U, 6)
X2      = HRC([F_pos, F_neg, U_score], [0.45, 0.40, 0.15])
```


In [169]:
p2 = data[1]

R_ind = p2["r"]
S_fit = p2["s"]
P_clar = p2["p"]

T_tech = bci(p2["n_features_available"], p2["n_features_needed"])

F_pos = hrc(
    values=[R_ind, S_fit, P_clar, T_tech],
    weights=[0.25, 0.30, 0.20, 0.25]
)

dissatisfaction_drag = p2["d"] * p2["cs"] 
custom_drag = p2["c"]

compliance_gap = 1.0 - p2["c2"]
compliance_penalty = edp(compliance_gap, 3.0)

F_neg = hrc(
    values=[1.0 - 0.4 * dissatisfaction_drag,
            1.0 - 0.25 * custom_drag,
            compliance_penalty],
    weights=[0.40, 0.25, 0.35]
)

U_score = ln_norm(p2["u"], 6)

X2 = hrc(
    values=[F_pos, F_neg, U_score],
    weights=[0.45, 0.40, 0.15]
)

print(f"X2 (Need & Product Fit): {X2:.6f}")


X2 (Need & Product Fit): 0.745826


## Parameter 3 — Authority & Decision Structure

Measures who is involved in the buying decision, how much authority they carry, and how aligned the internal stakeholders are.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `R` | Primary contact seniority. | `{0.2, 0.4, 0.6, 0.8, 1}` |
| `D` | Decision involvement of the contact. 0 = none, 0.5 = influencer, 1 = final decision-maker. | `{0, 0.5, 1}` |
| `Ac` | Contact authority = `R × D`. | `[0, 1]` |
| `N` | Total stakeholders identified in the purchasing decision. | integer |
| `O` | Organisational alignment — whether multiple departments agree on the need. | `[0, 1]` |
| `P` | Direct access to senior stakeholders. | `[0, 1]` |

### How it is computed

- **Stakeholder count** is normalised via `ln_norm(N, 5)` (log-based, diminishing returns past 5).
- **Organisational alignment** blends the stated `O` 50/50 with a BCI score from `n_stakeholders_supportive` out of `n_stakeholders_met`, anchoring stated alignment to observed stakeholder buy-in.
- All four components are aggregated via `hrc` (weights 0.30, 0.30, 0.25, 0.15).

### Composite Formula

```
Ac         = R × D
N_score    = ln_norm(N, 5)
O_combined = 0.5·O_stated + 0.5·BCI(supportive, met)
X3         = HRC([Ac, N_score, O_combined, P], [0.30, 0.30, 0.25, 0.15])
```


In [170]:
p3 = data[2]

A_c = p3["r"] * p3["d"]

N_stake = ln_norm(p3["n"], 5)

O_stated = p3["o"]
O_evidence = bci(p3["n_stakeholders_supportive"], p3["n_stakeholders_met"])
O_combined = 0.5 * O_stated + 0.5 * O_evidence

P_access = p3["p"]

X3 = hrc(
    values=[A_c, N_stake, O_combined, P_access],
    weights=[0.30, 0.30, 0.25, 0.15]
)

print(f"X3 (Authority & Decision): {X3:.6f}")


X3 (Authority & Decision): 0.769298


## Parameter 4 — Timeline, Urgency & Buying Stage

Measures how soon the prospect will decide and how much momentum the deal carries.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `T` | Days until expected final decision. | integer |
| `T1` | Whether a trigger event has occurred. | `{0, 1}` |
| `T2` | Days until the trigger event deadline. | integer |
| `Ep` | Evaluation process maturity — how structured the buying process is. | `[0, 1]` |
| `Ns` | Next step defined — whether a concrete next action is scheduled. | `{0, 1}` |
| `Cp` | Competing priorities — how much attention is divided elsewhere. 0 = top priority, 1 = heavily distracted. | `[0, 1]` |

### How it is computed

- **Timeline decay** (`T'`) uses exponential decay `exp(−0.015·T)`: deals at 0 days score ~1.0, at 45 days ~0.51, and distant deals asymptote toward 0 on a single smooth curve.
- **Trigger urgency** (`U_trig`) uses `T1 · exp(−0.01·T2)`, giving a smooth, always-positive urgency signal when a trigger exists (0 otherwise).
- **Process maturity** combines evaluation maturity and a defined next step: `0.5·Ep + 0.5·Ns`.
- **Competing priorities** enter the aggregation as the component `1 − Cp`, so distraction risk interacts with the other timing signals through the robust composite rather than acting as a blanket scalar.
- All four components are aggregated via `hrc` (weights 0.45, 0.15, 0.25, 0.15).

### Composite Formula

```
T'        = exp(−0.015·T)
U_trig    = T1 · exp(−0.01·T2)     [0 if no trigger]
P_process = 0.5·Ep + 0.5·Ns
X4        = HRC([T', U_trig, P_process, 1 − Cp], [0.45, 0.15, 0.25, 0.15])
```


In [171]:
p4 = data[3]

T_prime = edp(p4["t"], 0.015)

if p4["t1"] == 1:
    U_trig = edp(p4["t2"], 0.01) 
else:
    U_trig = 0.0

P_proc = 0.5 * p4["ep"] + 0.5 * p4["ns"]

T_core = hrc(
    values=[T_prime, U_trig, P_proc, 1.0 - p4["cp"]], 
    weights=[0.45, 0.15, 0.25, 0.15]
)

X4 = T_core

print(f"X4 (Timeline & Urgency): {X4:.6f}")


X4 (Timeline & Urgency): 0.643658


## Parameter 5 — Engagement Behaviour

Measures how actively the prospect is interacting across channels and whether that activity is accelerating or decaying.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `N1`–`N9` | Activity counts across nine channels: email opens (`N1`), email replies (`N2`), meetings held (`N3`), calls (`N4`), web sessions (`N5`), downloads (`N6`), pricing page visits (`N7`), demo requests (`N8`), social/community (`N9`). | integer each |
| `T` | Days since last meaningful interaction. | integer |
| `V` | Engagement velocity — ratio of last-14-day engagements to prior 14 days. | float |
| `C` | Channel diversity — number of distinct channels used. | integer |
| `M` | Negative signal count (unsubscribes, cancellations, no-shows, etc.). | integer |
| `weekly_engagement` | Time series of weekly engagement scores, oldest to newest, used for momentum detection. | list of floats |

### How it is computed

- **Per-channel normalisation**: each of the nine activity counts is independently normalised via `ln_norm(Ni, cap_i)` with channel-specific caps (e.g. 20 for email opens, 3 for demo requests). This prevents any single high-volume channel from dominating and ensures diminishing returns.
- **Composite engagement**: the nine normalised channel scores are aggregated via `hrc` with weights reflecting signal strength:
  - Demo requests and meetings: 0.18 each (strongest buying signals)
  - Pricing visits: 0.14
  - Email replies, downloads, calls: 0.10–0.12
  - Web sessions, email opens, social: 0.05–0.08
- **Recency decay**: `R_decay = exp(−0.033·T)`, applied multiplicatively to the composite engagement.
- **Momentum**: an EWMA (`λ = 0.85`) over `weekly_engagement` is compared to its simple mean; the positive difference (capped at 1.0) becomes the velocity bonus, scaled by 0.15, capturing genuine acceleration patterns.
- **Channel diversity**: `ln_norm(C, 5) × 0.10`.
- **Negative signals**: penalised via `exp(−0.3·M)`, applied as `√(N_penalty)` to soften impact while still discouraging repeated negative signals.
- **Final assembly**: core engagement (composite × decay), velocity bonus, and channel diversity are combined via `hrc` (weights 0.75, 0.15, 0.10), multiplied by `√(N_penalty)`, then passed through a sigmoid (`x0 = 0.3`, `k = 8`) for the final bounded score.

### Composite Formula

```
e_i          = ln_norm(N_i, cap_i)            for each channel i
E_composite  = HRC([e1 … e9], [0.05, 0.10, 0.18, 0.12, 0.08, 0.10, 0.14, 0.18, 0.05])
R_decay      = exp(−0.033·T)
V_bonus      = min(max(0, (EWMA − mean) / mean), 1.0) × 0.15
D_channel    = ln_norm(C, 5) × 0.10
N_penalty    = exp(−0.3·M)
E_core       = HRC([E_composite·R_decay, V_bonus, D_channel], [0.75, 0.15, 0.10])
X5           = sigmoid(E_core·√N_penalty, x0=0.3, k=8)
```


In [172]:
p5 = data[4]

e_email_open  = ln_norm(p5["n1"], 20) 
e_email_reply = ln_norm(p5["n2"], 10)  
e_meetings    = ln_norm(p5["n3"], 6) 
e_calls       = ln_norm(p5["n4"], 8)
e_web         = ln_norm(p5["n5"], 15) 
e_downloads   = ln_norm(p5["n6"], 8) 
e_pricing     = ln_norm(p5["n7"], 5) 
e_demo_req    = ln_norm(p5["n8"], 3) 
e_social      = ln_norm(p5["n9"], 5) 


E_composite = hrc(
    values=[e_email_open, e_email_reply, e_meetings, e_calls,
            e_web, e_downloads, e_pricing, e_demo_req, e_social],
    weights=[0.05, 0.10, 0.18, 0.12, 0.08, 0.10, 0.14, 0.18, 0.05]
)

R_decay = edp(p5["t"], 0.033)

weekly = p5["weekly_engagement"]
ewma_val = ewma_mean(weekly, lam=0.85)
simple_mean = np.mean(weekly)

V_bonus = max(0.0, (ewma_val - simple_mean) / max(simple_mean, 0.01))
V_score = min(V_bonus, 1.0) * 0.15

D_channel = ln_norm(p5["c"], 5) * 0.10

N_penalty = edp(p5["m"], 0.3)

E_core = hrc(
    values=[E_composite * R_decay, V_score, D_channel],
    weights=[0.75, 0.15, 0.10]
)

X5_raw = E_core * np.sqrt(N_penalty)
X5 = sigmoid_transform(X5_raw, x0=0.3, k=8.0)

print(f"X5 (Engagement Behaviour): {X5:.6f}")


X5 (Engagement Behaviour): 0.727083


## Parameter 6 — Company & Market Fit

Measures Ideal Customer Profile (ICP) alignment — how closely the prospect matches the type of customer our company serves best, and how practical the business relationship would be.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `Seg` | Customer segment match. | `[0, 1]` |
| `Emp` | Employee size fit relative to ideal. | `[0, 1]` |
| `Rev` | Annual revenue fit relative to ideal range. | `[0, 1]` |
| `Tech` | Technology stack compatibility. | `[0, 1]` |
| `Geo` | Geographic alignment with active service regions. | `[0, 1]` |
| `Gro` | Company growth trajectory. | `[0, 1]` |
| `F` | Financial health / credit risk. | `[0, 1]` |
| `D` | Digital adoption readiness. | `[0, 1]` |
| `L` | Language and culture compatibility. | `[0, 1]` |

### How it is computed

- The nine ICP variables are aggregated via `hrc` (weights 0.20, 0.10, 0.10, 0.20, 0.05, 0.10, 0.10, 0.10, 0.05), providing robustness against any single outlier dimension.
- The result is passed through a sigmoid (`x0 = 0.70`, `k = 8`) to sharpen discrimination around the "good-fit" boundary: prospects solidly above 0.70 are rewarded, those below are penalised more steeply than a linear score would reflect.

### Composite Formula

```
X6_raw = HRC([Seg, Emp, Rev, Tech, Geo, Gro, F, D, L],
             [0.20, 0.10, 0.10, 0.20, 0.05, 0.10, 0.10, 0.10, 0.05])
X6     = sigmoid(X6_raw, x0=0.70, k=8)
```


In [159]:

p6 = data[5]

X6_raw = hrc(
    values=[p6["seg"], p6["emp"], p6["rev"], p6["tech"],
            p6["geo"], p6["gro"], p6["f"], p6["d"], p6["l"]],
    weights=[0.20, 0.10, 0.10, 0.20, 0.05, 0.10, 0.10, 0.10, 0.05]
)

X6 = sigmoid_transform(X6_raw, x0=0.70, k=8.0)

print(f"X6 (Company & Market Fit): {X6:.6f}")


X6 (Company & Market Fit): 0.720586


## Parameter 7 — Lead Source Quality

Evaluates where the lead came from, how that source channel historically performs, and how much useful data we captured at entry.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `Q` | Source channel quality — historical effectiveness of the entry channel (e.g. 0.15 for purchased lists up to 0.90 for customer referrals). | `[0, 1]` |
| `P` | Campaign/asset quality — performance of the specific campaign relative to best-performing. | `[0, 1]` |
| `R` | Data richness at entry. 0 = email only, 1 = full profile with company, role, phone, stated interest. | `[0, 1]` |
| `S` | Inbound vs. outbound. 0.4 = outbound (we initiated), 1.0 = inbound (they came to us). | `{0.4, 1}` |

### How it is computed

- **Source channel quality** (`Q`) blends the stated quality 50/50 with a BCI score from `n_converted_from_source` out of `n_leads_from_source`. A channel with a strong conversion history on a decent sample validates the stated quality; a thin sample keeps the BCI conservative and pulls the score down.
- The four components are aggregated via `hrc` (weights 0.55, 0.15, 0.10, 0.20).

### Composite Formula

```
Q  = 0.5·Q_stated + 0.5·BCI(converted, total_leads)
X7 = HRC([Q, P, R, S], [0.55, 0.15, 0.10, 0.20])
```


In [174]:
p7 = data[6]

Q_stated = p7["q"]
Q_evidence = bci(p7["n_converted_from_source"], p7["n_leads_from_source"])
Q = 0.5 * Q_stated + 0.5 * Q_evidence

P_camp = p7["p"]
R_data = p7["r"]
S_dir  = p7["s"]

X7 = hrc(
    values=[Q, P_camp, R_data, S_dir],
    weights=[0.55, 0.15, 0.10, 0.20]
)

print(f"X7 (Lead Source Quality): {X7:.6f}")


X7 (Lead Source Quality): 0.677632


## Parameter 8 — Competitive Landscape

Evaluates competitive pressure — how many alternatives the prospect is considering, our historical win rate, incumbent strength, differentiation, and switching friction.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `N` | Number of competing vendors being evaluated. | integer |
| `W` | Historical win rate against these competitors in this segment. | `[0, 1]` |
| `S` | Incumbent strength — how entrenched the current vendor is. 0 = weak/none, 1 = deeply entrenched. | `[0, 1]` |
| `D` | Differentiation — how clearly we stand out from alternatives. | `[0, 1]` |
| `C` | Switching cost for the prospect. 0 = trivial, 1 = massive. | `[0, 1]` |
| `K` | Sole vendor flag. 1 = we are the only vendor being considered. | `{0, 1}` |

### How it is computed

- If `K = 1` (sole vendor), the score is set directly to **1.0**.
- Otherwise:
  - **Competitor count** is suppressed via exponential decay `exp(−0.18·N)`, penalising the first few additional competitors steeply and flattening for crowded evaluations.
  - **Win rate** is computed via BCI from `n_wins` out of `n_competitive_deals`, grounding it in actual outcomes and staying conservative on small samples.
  - **Competitive drag** (incumbent strength + switching cost) uses `exp(−1.0·(0.5·S + 0.5·C))`: moderate friction has limited impact, extreme friction is nearly disqualifying.
  - All four components are aggregated via `hrc` (weights 0.25, 0.30, 0.20, 0.25).

### Composite Formula

```
If K = 1:  X8 = 1.0
Else:
    N_comp = exp(−0.18·N)
    W_bci  = BCI(wins, competitive_deals)
    Cd     = exp(−1.0·(0.5·S + 0.5·C))
    X8     = HRC([N_comp, W_bci, Cd, D], [0.25, 0.30, 0.20, 0.25])
```


In [175]:

p8 = data[7]

if p8["k"] == 1:
    X8 = 1.0
else:
    N_comp = edp(p8["n"], 0.18)

    W_bci = bci(p8["n_wins"], p8["n_competitive_deals"])

    friction = 0.5 * p8["s"] + 0.5 * p8["c"]
    Cd = edp(friction, 1.0)

    D_diff = p8["d"]

    X8 = hrc(
        values=[N_comp, W_bci, Cd, D_diff],
        weights=[0.25, 0.30, 0.20, 0.25]
    )

print(f"X8 (Competitive Landscape): {X8:.6f}")

X8 (Competitive Landscape): 0.620278


## Parameter 9 — Relationship & Trust Equity

Measures the strength and depth of the existing relationship between our organisation and the prospect — familiarity, trust signals, executive sponsorship, and any negative history.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `Pb` | Previous business relationship. 0 = net new, 0.5 = lapsed/informal, 1.0 = active/recent customer. | `{0, 0.5, 1}` |
| `Rt` | Relationship tenure in months. | integer |
| `Nps` | Prospect sentiment. −1 = detractor, 0 = passive, 1 = promoter. New leads default to 0. | `[−1, 1]` |
| `Es` | Executive sponsor relationship. 0 = none, 0.5 = acquaintance, 1 = strong personal relationship. | `{0, 0.5, 1}` |
| `T` | Trust indicators — observable signals of transparency and collaboration. | `[0, 1]` |
| `Rc` | Reference customer — whether we can point to a relevant success story known to this prospect. | `[0, 1]` |
| `N` | Previous negative experience severity. 0 = none, 0.5 = minor, 1 = serious. | `{0, 0.5, 1}` |

### How it is computed

- **Relationship tenure** uses `Pb · ln_norm(Rt, 36)` — log-based normalisation with diminishing returns past 36 months, gated by whether a prior relationship exists at all.
- **Trust** sub-score aggregates `[Es, T, Rc, max(Nps, 0)]` via `hrc` (weights 0.30, 0.30, 0.20, 0.20).
- **Negative experience** penalty uses exponential decay `exp(−1.5·N)`: a minor issue (`N = 0.5`) still allows most of the score through, while a serious issue (`N = 1`) is nearly devastating.
- Tenure and trust are combined via `hrc` (weights 0.40, 0.60) before the negative penalty is applied multiplicatively.

### Composite Formula

```
R_ten   = Pb · ln_norm(Rt, 36)
R_trust = HRC([Es, T, Rc, max(Nps, 0)], [0.30, 0.30, 0.20, 0.20])
R_neg   = exp(−1.5·N)
X9      = R_neg · HRC([R_ten, R_trust], [0.40, 0.60])
```


In [162]:
p9 = data[8]

R_ten = p9["pb"] * ln_norm(p9["rt"], 36)

R_trust = hrc(
    values=[p9["es"], p9["t"], p9["rc"], max(p9["nps"], 0)],
    weights=[0.30, 0.30, 0.20, 0.20]
)

R_neg = edp(p9["n"], 1.5)

X9 = R_neg * hrc(
    values=[R_ten, R_trust],
    weights=[0.40, 0.60]
)

print(f"X9 (Relationship & Trust): {X9:.6f}")


X9 (Relationship & Trust): 0.693671


## Parameter 10 — Strategic & Lifetime Value

Evaluates the long-term value of the account beyond the immediate deal — lifetime revenue potential, strategic market impact, expansion likelihood, and churn risk.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `LTV` | Estimated lifetime value of the customer relationship. | currency |
| `CAC` | Customer acquisition cost. | currency |
| `CAC'` | LTV / CAC ratio. | float |
| `Cs` | Cross-sell / up-sell opportunity after initial purchase. | `[0, 1]` |
| `B` | Brand value — reputational benefit of winning this account. | `[0, 1]` |
| `M` | Market expansion — whether this win opens a new market. 0 = no, 0.5 = strengthens existing, 1 = new market entry. | `{0, 0.5, 1}` |
| `E` | Expansion revenue probability — likelihood the account grows post-purchase. | `[0, 1]` |
| `C` | Churn risk — predicted probability of cancellation or non-renewal. | `[0, 1]` |

### How it is computed

- **LTV/CAC ratio** is passed through a sigmoid `σ(CAC', x0=3.0, k=1.5)`, rewarding healthy return-on-acquisition.
- **Cross-sell** score blends the stated value 50/50 with a BCI from `n_expanded` out of `n_similar_accounts`, grounding the estimate in historical expansion data for comparable accounts.
- **Strategic value** aggregates `[B, M, E]` via `hrc` (weights 0.30, 0.40, 0.30).
- **Churn penalty** uses exponential decay `exp(−1.2·C)`: moderate risk (`C = 0.3`) barely hurts, high risk (`C = 0.8`) slashes the score.
- The three value components are combined via `hrc` (weights 0.35, 0.25, 0.40) before the churn penalty is applied multiplicatively.

### Composite Formula

```
V_ltv   = sigmoid(LTV/CAC, x0=3.0, k=1.5)
Cs      = 0.5·Cs_stated + 0.5·BCI(expanded, similar_accounts)
V_strat = HRC([B, M, E], [0.30, 0.40, 0.30])
V_churn = exp(−1.2·C)
X10     = V_churn · HRC([V_ltv, Cs, V_strat], [0.35, 0.25, 0.40])
```


In [176]:
p10 = data[9]

cac_ratio = p10["ltv"] / p10["cac"]
V_ltv = sigmoid_transform(cac_ratio, x0=3.0, k=1.5)

Cs_stated = p10["cs"]
Cs_evidence = bci(p10["n_expanded"], p10["n_similar_accounts"])
Cs = 0.5 * Cs_stated + 0.5 * Cs_evidence

V_strat = hrc(
    values=[p10["b"], p10["m"], p10["e"]],
    weights=[0.30, 0.40, 0.30]
)

V_churn = edp(p10["c"], 1.2)

X10 = V_churn * hrc(
    values=[V_ltv, Cs, V_strat],
    weights=[0.35, 0.25, 0.40]
)

print(f"X10 (Strategic & Lifetime Value): {X10:.6f}")

X10 (Strategic & Lifetime Value): 0.439952


## Hard Knockout Gates

Seven binary gates that can instantly zero the entire lead score. If any single gate fails, the lead is disqualified regardless of how strong the other parameters are.

| Gate | Condition that fails the gate (sets it to 0) |
|---|---|
| `G1` | No realistic financial ability — confirmed budget = 0 **and** no identified funding path or budget cycle. |
| `G2` | Communication compliance violation — contact has opted out, unsubscribed, or sent a cease-and-desist. |
| `G3` | Pipeline integrity — lead is a duplicate record or already an active customer (should route to account management). |
| `G4` | Sanctions / compliance blacklist — company or individual appears on a sanctions list, trade restriction, or compliance blacklist. |
| `G5` | Geographic restriction — company is in a country or region we legally cannot sell to or support. |
| `G6` | Prohibited industry — company operates in an industry our organisation has a formal policy against serving. |
| `G7` | Technical infeasibility — after assessment, our product fundamentally cannot serve the prospect's core need and no roadmap path exists. |

Each gate is encoded as `"f"` (pass) or `"t"` (triggered / fail) in the `gates` list. The composite gate is:

```
G = G1 × G2 × G3 × G4 × G5 × G6 × G7
```

Any single 0 makes `G = 0`, killing the final score.


In [177]:
g1=[]
for i in range(len(gates)):
    h=gates[i]
    if h == 't':
        g1.append(0)
    else:
        g1.append(1)
G=1
for h in g1:
    G=G*h
print(G)

1


## Deal-Level Modifiers

Five multiplicative factors that adjust the final score for practical deal-execution realities. Unlike gates (binary), modifiers are continuous scalars that reduce the score proportionally.

| Modifier | What it captures | Range |
|---|---|---|
| `M1` | Data reliability — are key deal details (budget, company size, contacts, requirements) confirmed or estimated/missing? `0.5 + 0.5·data_completeness`. | `[0.5, 1.0]` |
| `M2` | Sales capacity — does the team have bandwidth and the right rep available? 1.0 = yes, 0.7 = stretched, 0.5 = no rep in region/segment. | `[0.5, 1.0]` |
| `M3` | Legal / contract complexity — severity of non-standard terms (liability clauses, IP terms, special agreements). 1.0 = standard, down to 0.6 for extreme requirements. | `[0.6, 1.0]` |
| `M4` | Payment / financial risk — currency fluctuation, long payment cycles, high-risk markets. 1.0 = low risk, down to 0.7. | `[0.7, 1.0]` |
| `M5` | Execution complexity — multi-location, language barriers, heavy customisation, complex approval chains. 1.0 = simple, down to 0.6 for very complex deals. | `[0.6, 1.0]` |

```
M = M1 × M2 × M3 × M4 × M5
```


In [178]:
a=[]
for i in range(len(modifiers)):
    h=modifiers[i]
    a.append(h)
M=1
for i in range(5):
    M=M*a[i]
print(M)


0.9


## Final Score — Choquet Integral Aggregation

The final score combines the ten parameter scores using a **Choquet integral** over a non-additive (fuzzy) measure, which captures interaction effects between parameters that a simple weighted sum cannot express.

### Why a Choquet integral?

A weighted sum assumes each parameter contributes independently. In reality, parameters interact:

- **Synergies** — a lead with strong financials (`X1`) and high urgency (`X4`) is worth more than the sum of those scores would suggest: budget plus urgency together signal an imminent close.
- **Redundancies** — high company fit (`X6`) and high lead-source quality (`X7`) partially overlap in what they tell us, so stacking both shouldn't double-count the confidence.

The Choquet integral handles this with a **capacity** (fuzzy measure) `μ` over all subsets of parameters. For tractability, we use a **2-additive capacity**:

- **Singleton capacities** `a_i` — the base importance of each parameter (analogous to weights).
- **Pairwise interaction coefficients** `a_ij` — positive values create synergy (the pair together is worth more), negative values create redundancy (the pair together is worth less).

### Singleton capacities

| Parameter | `a_i` | Rationale |
|---|---|---|
| `X1` — Financial Qualification | 0.155 | Strong single predictor — no budget, no deal. |
| `X2` — Need & Product Fit | 0.125 | Core value proposition alignment. |
| `X3` — Authority & Decision Structure | 0.200 | Highest weight — access to decision-makers is the #1 accelerator. |
| `X4` — Timeline & Urgency | 0.125 | Timing drives pipeline velocity. |
| `X5` — Engagement Behaviour | 0.120 | Behavioural intent signals. |
| `X6` — Company & Market Fit | 0.060 | Important but largely static / pre-qualification. |
| `X7` — Lead Source Quality | 0.030 | Entry quality — diminishes as engagement data accumulates. |
| `X8` — Competitive Landscape | 0.055 | External pressure factor. |
| `X9` — Relationship & Trust | 0.055 | Relationship equity. |
| `X10` — Strategic & Lifetime Value | 0.075 | Long-term account potential. |

### Pairwise interactions

| Pair | Coefficient | Interpretation |
|---|---|---|
| (`X1`, `X4`) Financial + Timeline | +0.025 | Budget + urgency → imminent close (synergy). |
| (`X1`, `X3`) Financial + Authority | +0.020 | Budget holder with decision power → strong (synergy). |
| (`X2`, `X5`) Need + Engagement | +0.015 | Clear need + active engagement → high intent (synergy). |
| (`X3`, `X4`) Authority + Timeline | +0.015 | Decision-maker + deadline → deal momentum (synergy). |
| (`X8`, `X1`) Competitive + Financial | +0.015 | Competitive advantage + budget → defensible deal (synergy). |
| (`X9`, `X10`) Relationship + Strategic | +0.010 | Trust + long-term value → partnership potential (synergy). |
| (`X5`, `X9`) Engagement + Relationship | −0.010 | High engagement from an existing relationship is partly redundant. |
| (`X6`, `X7`) Company Fit + Source | −0.010 | Both measure "right kind of lead" — some overlap. |
| (`X2`, `X6`) Need + Company Fit | −0.005 | Product fit and ICP fit correlate — slight redundancy. |

### Pre-processing before the integral

- Three parameter scores (`X4`, `X6`, `X9` — indices 3, 5, 8) receive an additional sigmoid transform (`x0 = 0.50`, `k = 8.0`) before entering the integral. These tend to cluster in a narrow mid-range, and the sigmoid spreads them for better discrimination.
- All scores are clamped to `[0, 1]` before aggregation.

### Final formula

```
C_μ   = ChoquetIntegral(X_final, a, a_ij)
μ(N)  = mu(full set)          [normalisation constant]
S     = 100 × G × M × (C_μ / μ(N))
```

The result is a score between 0 and 100, where:

- **`G`** (gates) can zero it instantly,
- **`M`** (modifiers) scales it for deal-execution friction, and
- **`C_μ / μ(N)`** is the Choquet-normalised composite of all ten parameters.


In [181]:
def mu(S, a, a_ij):
    value = sum(a[i] for i in S)
    for (i, j), coeff in a_ij.items():
        if i in S and j in S:
            value += coeff
    return value
    
def choquet_integral(scores, a, a_ij):
    n = len(scores)
    sigma = sorted(range(n), key=lambda i: scores[i])
    integral = 0.0
    for rank in range(n):
        coalition = set(sigma[rank:])
        mu_val = mu(coalition, a, a_ij)
        if rank == 0:
            delta = scores[sigma[rank]]
        else:
            delta = scores[sigma[rank]] - scores[sigma[rank - 1]]
        integral += delta * mu_val
    return integral
    
X_all = [X1, X2, X3, X4, X5, X6, X7, X8, X9, X10]

sigmoid_indices = {3, 5, 8}  
X_final = [
    sigmoid_transform(x, x0=0.50, k=8.0) if i in sigmoid_indices else x
    for i, x in enumerate(X_all)
]
X_final = [max(0.0, min(1.0, x)) for x in X_final]

a_vals = {
    0: 0.155, 1: 0.125, 2: 0.200, 3: 0.125, 4: 0.120,  
    5: 0.060, 6: 0.030, 7: 0.055, 8: 0.055, 9: 0.075,   
}

a_ij_vals = {
    (0, 3): +0.025,
    (0, 2): +0.020, 
    (1, 4): +0.015,  
    (2, 3): +0.015,  
    (7, 0): +0.015, 
    (8, 9): +0.010, 
    (4, 8): -0.010,  
    (5, 6): -0.010,   
    (1, 5): -0.005,   
}

C_mu = choquet_integral(X_final, a_vals, a_ij_vals)
mu_N = mu(set(range(10)), a_vals, a_ij_vals)

S = 100.0 * G * M * (C_mu / mu_N)

print(f"\n{'='*100}")
print(f"  FINAL LEAD SCORE: {S:.2f} / 100")
print(f"{'='*100}")



  FINAL LEAD SCORE: 65.91 / 100
